In [1]:
# Cell 1 — Imports + paths (edit RUN_DIR, then run)  

import os  # 02.23.2026 CHANGED: keep basics only
from pathlib import Path  # 02.23.2026 CHANGED: safer paths
import gc  # 02.23.2026 CHANGED: explicit cleanup for big arrays

import numpy as np  # 02.23.2026 CHANGED
import pandas as pd  # 02.23.2026 CHANGED
import zarr  # 02.23.2026 CHANGED

import matplotlib.pyplot as plt  # 02.23.2026 CHANGED
from datetime import datetime  # 02.23.2026 CHANGED
import json  # 03.31.2026 ADDED: save experiment contexts and combined run metadata
from tqdm.auto import tqdm  # 02.23.2026 CHANGED

from scipy.ndimage import zoom  # 02.23.2026 CHANGED: response-map downsampling (no fallbacks)

from sklearn.ensemble import RandomForestClassifier  # 02.23.2026 CHANGED
from sklearn.model_selection import StratifiedKFold  # 02.23.2026 CHANGED
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay  # 02.23.2026 CHANGED

from sklearn.linear_model import LogisticRegression  # 02.23.2026 CHANGED: add MLR baseline
from sklearn.pipeline import make_pipeline  # 02.23.2026 CHANGED: bundle scaling + model
from sklearn.preprocessing import StandardScaler  # 02.23.2026 CHANGED: scale features for MLR
from sklearn.svm import SVC  # 02.26.2026 CHANGED: add SVM option

import joblib  # 02.28.2026 ADDED: save/load fitted sklearn models (RF/MLR/SVM)  

# ----------------------------
# Your new run (source-of-truth paths)
RUN_DIR = Path("export_retina_20260313_230715")  # 02.23.2026 CHANGED: new dataset run folder
ZARR_PATH = RUN_DIR / "dataset.zarr"  # 02.23.2026 CHANGED
META_PATH = RUN_DIR / "metadata.csv"  # 02.23.2026 CHANGED

assert ZARR_PATH.exists(), "Missing Zarr: " + str(ZARR_PATH)  # 02.23.2026 CHANGED: fail fast
assert META_PATH.exists(), "Missing CSV : " + str(META_PATH)  # 02.23.2026 CHANGED: fail fast

In [2]:
# Cell 1b — Config: experiment list + shared hyperparameters  # 03.31.2026 CHANGED: move run control into explicit experiment specs

RUN_MODE = "all"  # 03.31.2026 CHANGED: default model bundle used when an experiment includes all supported models

if RUN_MODE.lower() not in {"rf", "mlr", "svm", "all"}:  # 03.31.2026 CHANGED: validate the shared default model selector
    raise ValueError('RUN_MODE must be "rf", "mlr", "svm", or "all" in this notebook version.')  # 03.31.2026 CHANGED: clear allowed values for the shared default model selector

if RUN_MODE.lower() == "all":  # 03.31.2026 CHANGED: expand the shared default model bundle into all three model names
    MODELS_TO_RUN = ["RF", "MLR", "SVM"]  # 03.31.2026 CHANGED: default model bundle for experiments that use all models
else:  # 03.31.2026 CHANGED: keep one-model mode available through the shared default selector
    MODELS_TO_RUN = [RUN_MODE.upper()]  # 03.31.2026 CHANGED: convert the shared default selector into one uppercase model name

DEFAULT_SEED = 12  # 03.31.2026 ADDED: shared seed you can reuse across experiment dictionaries
USE_SAVED_SPLIT = True  # 03.31.2026 CHANGED: reuse an exact saved split when an experiment config matches a saved split file

EXPERIMENTS = [  # 03.31.2026 ADDED: one dictionary per experiment; the number of experiments is len(EXPERIMENTS)
    {  # 03.31.2026 ADDED: edit this dictionary or add more dictionaries to create more experiments
        "name": "experiment_01",  # 03.31.2026 ADDED: unique experiment folder name inside the batch output directory
        "groups": ["Classic4", "Classic4_plus_s"],  # 03.31.2026 ADDED: keep groups together here, or place one group per dictionary to separate groups across experiments
        "models": list(MODELS_TO_RUN),  # 03.31.2026 ADDED: keep models together here, or place one model per dictionary to separate models across experiments
        "seed": int(DEFAULT_SEED),  # 03.31.2026 ADDED: set the experiment-specific random seed here
        "merge_gray_classes": False,  # 03.31.2026 ADDED: set True to merge gray_d and gray_l into one gray class for this experiment
        "subset_max_per_class": None,  # 03.31.2026 ADDED: set an integer to use a balanced smaller subset per class, or leave None for the full dataset
    },  # 03.31.2026 ADDED: end experiment_01 configuration
]  # 03.31.2026 ADDED: add more dictionaries above to run more experiments in one notebook pass

N_EXPERIMENTS = len(EXPERIMENTS)  # 03.31.2026 ADDED: explicit notebook-visible count of configured experiments

MAX_DEPTH = 5  # 03.31.2026 CHANGED: shared RF hyperparameter used by every RF fit in this notebook run
N_EST = 200  # 03.31.2026 CHANGED: shared RF hyperparameter used by every RF fit in this notebook run
RF_N_JOBS = 1  # 03.31.2026 CHANGED: keep RF parallelism modest to reduce memory spikes

MLR_MAX_ITER = 200  # 03.31.2026 CHANGED: shared MLR hyperparameter used by every MLR fit in this notebook run
MLR_C = 0.3  # 03.31.2026 CHANGED: shared MLR hyperparameter used by every MLR fit in this notebook run
MLR_TOL = 1e-2  # 03.31.2026 CHANGED: shared MLR hyperparameter used by every MLR fit in this notebook run

SVM_KERNEL = "rbf"  # 03.31.2026 CHANGED: shared SVM hyperparameter used by every SVM fit in this notebook run
SVM_C = 1.0  # 03.31.2026 CHANGED: shared SVM hyperparameter used by every SVM fit in this notebook run
SVM_GAMMA = "scale"  # 03.31.2026 CHANGED: shared SVM hyperparameter used by every SVM fit in this notebook run
SVM_PROBABILITY = False  # 03.31.2026 CHANGED: keep probability mode off unless you explicitly need it

print("MODELS_TO_RUN default:", MODELS_TO_RUN)  # 03.31.2026 ADDED: show the shared default model bundle
print("Configured experiments:", N_EXPERIMENTS)  # 03.31.2026 ADDED: show the number of experiment dictionaries that will run

for _exp in EXPERIMENTS:  # 03.31.2026 ADDED: print a concise summary of every configured experiment
    print(_exp["name"], {"groups": _exp["groups"], "models": _exp["models"], "seed": _exp["seed"], "merge_gray_classes": _exp["merge_gray_classes"], "subset_max_per_class": _exp["subset_max_per_class"]})  # 03.31.2026 ADDED: display the key controls for each experiment

MODELS_TO_RUN: ['RF', 'MLR', 'SVM']


In [3]:
# Cell 2 — Load Zarr + metadata  # 02.23.2026 CHANGED

# Open Zarr group (read-only)
root = zarr.open_group(str(ZARR_PATH), mode="r")  # 02.23.2026 CHANGED

# Required Zarr arrays
imgs_z = root["imgs"]  # 02.23.2026 CHANGED: (N,H,W,3)
outs_fill_z = root["derived"]["outs_fill"]  # 02.23.2026 CHANGED: (N,K,H,W) — RF uses this

# Required Zarr attrs
keys_order = list(root.attrs["RESPONSE_KEYS"])  # 02.23.2026 CHANGED: canonical channel order (length K)

# Basic shapes
N = int(imgs_z.shape[0])  # 02.23.2026 CHANGED
H0 = int(outs_fill_z.shape[2])  # 02.23.2026 CHANGED
W0 = int(outs_fill_z.shape[3])  # 02.23.2026 CHANGED
K = int(outs_fill_z.shape[1])  # 02.23.2026 CHANGED

assert outs_fill_z.shape[0] == N, "Mismatch: imgs and outs_fill have different N"  # 02.23.2026 CHANGED
assert len(keys_order) == K, "Mismatch: len(RESPONSE_KEYS) != outs_fill.shape[1]"  # 02.23.2026 CHANGED

# Load metadata and link rows by stim_ID
meta = pd.read_csv(META_PATH)  # 02.23.2026 CHANGED
meta = meta.set_index("stim_ID").sort_index()  # 02.23.2026 CHANGED: stim_ID is the join key

# --- Clean, explicit alignment check (no fallbacks) ---
expected = np.arange(N, dtype=np.int64)  # 02.23.2026 CHANGED
got = meta.index.to_numpy(dtype=np.int64)  # 02.23.2026 CHANGED
if not np.array_equal(got, expected):  # 02.23.2026 CHANGED
    msg = (
        "Expected metadata.index (stim_ID) to be exactly [0,1,...,N-1] to match Zarr row order.\n"
        + "Found stim_ID range: [{}, {}] with {} rows; expected N={}.\n".format(int(got.min()), int(got.max()), len(got), N)
        + "Clean fix: re-export so stim_ID increments from 0 and writes Zarr rows in the same order,\n"
        + "OR re-write dataset.zarr so its first axis is ordered by stim_ID (no implicit remapping in this RF notebook)."
    )
    raise ValueError(msg)  # 02.23.2026 CHANGED

print("Loaded:", {"N": N, "K": K, "H0": H0, "W0": W0})
print("First 10 RESPONSE_KEYS:", keys_order[:10])
print("Metadata columns:", list(meta.columns))


Loaded: {'N': 4710, 'K': 9, 'H0': 1000, 'W0': 1000}
First 10 RESPONSE_KEYS: ['l_on', 'l_off', 'm_on', 'm_off', 'l_h2_on', 'm_h2_on', 'l_h2_off', 'm_h2_off', 's_on']
Metadata columns: ['true_label', 'bg_hue', 'bg_r', 'bg_g', 'bg_b', 'obj_r', 'obj_g', 'obj_b', 'bg_int', 'obj_int', 'delta_int', 'bg_sat', 'obj_sat', 'delta_sat', 'hsv_bg', 'hsv_obj', 'delta_hsv']


In [4]:
# Cell 2b — Optional: load a saved RESULTS DIRECTORY (already-saved PNG/TXT/CSV outputs)  # 03.31.2026 CHANGED: support the new multi-experiment save layout

import re  # 02.26.2026 ADDED: filename parsing for saved-results directories

LOAD_SAVED_RESULTS_DIR = False  # 03.31.2026 CHANGED: set True to browse a previously saved v05 results folder
SAVED_RESULTS_DIR = Path("svm_mlr_rf_eval_batch_YYYYMMDD_HHMMSS")  # 03.31.2026 CHANGED: point this at a saved batch output folder when LOAD_SAVED_RESULTS_DIR is True

USE_LOADED_RESULTS = False  # 03.31.2026 CHANGED: later cells use this switch to skip new training and browse saved outputs instead
SAVED_EXPERIMENT_DIRS = {}  # 03.31.2026 ADDED: map saved experiment name -> saved experiment folder path
SAVED_EXPERIMENT_NAMES = []  # 03.31.2026 ADDED: ordered experiment names discovered in the saved results folder
SAVED_MODELS_BY_EXPERIMENT = {}  # 03.31.2026 ADDED: map saved experiment name -> available saved models
SAVED_GROUPS_BY_EXPERIMENT = {}  # 03.31.2026 ADDED: map saved experiment name -> available saved groups
SAVED_COMBINED_SUMMARY_PATH = None  # 03.31.2026 ADDED: combined summary file path for a saved batch folder
COMBINED_EVAL_SUMMARY_DF = None  # 03.31.2026 ADDED: combined saved summary dataframe when one is available

if LOAD_SAVED_RESULTS_DIR:  # 03.31.2026 CHANGED: inspect the saved batch directory now
    if not SAVED_RESULTS_DIR.exists():  # 03.31.2026 CHANGED: fail clearly if the requested saved directory is missing
        raise FileNotFoundError("Missing results directory: " + str(SAVED_RESULTS_DIR))  # 03.31.2026 CHANGED: clear error for a missing saved directory

    OUTDIR = SAVED_RESULTS_DIR  # 03.31.2026 CHANGED: point later viewer cells at the saved batch root
    SAVED_COMBINED_SUMMARY_PATH = OUTDIR / "combined_eval_summary.csv"  # 03.31.2026 ADDED: expected combined summary path for a saved batch run

    if SAVED_COMBINED_SUMMARY_PATH.exists():  # 03.31.2026 ADDED: load the saved combined summary when present
        COMBINED_EVAL_SUMMARY_DF = pd.read_csv(SAVED_COMBINED_SUMMARY_PATH)  # 03.31.2026 ADDED: cached combined summary dataframe for display later

    candidate_dirs = sorted([p for p in OUTDIR.iterdir() if p.is_dir() and (((p / "run_context.json").exists()) or ((p / "eval_summary.csv").exists()))], key=lambda p: p.name)  # 03.31.2026 ADDED: detect experiment folders by their saved context or summary files

    if len(candidate_dirs) == 0:  # 03.31.2026 ADDED: treat the root as one saved run when no experiment subfolders were found
        candidate_dirs = [OUTDIR]  # 03.31.2026 ADDED: single-run fallback for saved browsing inside this notebook version

    for exp_dir in candidate_dirs:  # 03.31.2026 ADDED: discover saved experiment names, models, and groups
        exp_name = exp_dir.name if exp_dir != OUTDIR else "saved_run"  # 03.31.2026 ADDED: stable experiment label for the root fallback case
        models = set()  # 03.31.2026 ADDED: collect saved model names for this experiment folder
        groups = set()  # 03.31.2026 ADDED: collect saved group names for this experiment folder

        for p in exp_dir.iterdir():  # 03.31.2026 ADDED: scan files inside the saved experiment folder
            name = p.name  # 03.31.2026 ADDED: convenience variable for filename parsing

            m = re.match(r"(RF|MLR|SVM)_report_(.+)\.txt$", name)  # 03.31.2026 ADDED: report filenames encode model and group
            if m:  # 03.31.2026 ADDED: store model and group discovered from a saved report
                models.add(m.group(1))  # 03.31.2026 ADDED: add the saved model name for this experiment
                groups.add(m.group(2))  # 03.31.2026 ADDED: add the saved group name for this experiment
                continue  # 03.31.2026 ADDED: continue scanning other files in the folder

            m = re.match(r"(RF|MLR|SVM)_cm_pct_(.+?)_(annotated|blank)_acc.*\.png$", name)  # 03.31.2026 ADDED: confusion-matrix PNG filenames encode model and group
            if m:  # 03.31.2026 ADDED: store model and group discovered from a saved confusion matrix
                models.add(m.group(1))  # 03.31.2026 ADDED: add the saved model name for this experiment
                groups.add(m.group(2))  # 03.31.2026 ADDED: add the saved group name for this experiment
                continue  # 03.31.2026 ADDED: continue scanning other files in the folder

            m = re.match(r"misclassified_(RF|MLR|SVM)_(.+)\.csv$", name)  # 03.31.2026 ADDED: misclassified CSV filenames encode model and group
            if m:  # 03.31.2026 ADDED: store model and group discovered from a misclassified CSV
                models.add(m.group(1))  # 03.31.2026 ADDED: add the saved model name for this experiment
                groups.add(m.group(2))  # 03.31.2026 ADDED: add the saved group name for this experiment
                continue  # 03.31.2026 ADDED: continue scanning other files in the folder

        SAVED_EXPERIMENT_DIRS[exp_name] = exp_dir  # 03.31.2026 ADDED: remember where this saved experiment lives on disk
        SAVED_EXPERIMENT_NAMES.append(exp_name)  # 03.31.2026 ADDED: preserve the order of saved experiment names for dropdowns later
        SAVED_MODELS_BY_EXPERIMENT[exp_name] = sorted(models)  # 03.31.2026 ADDED: save the model options for this experiment
        SAVED_GROUPS_BY_EXPERIMENT[exp_name] = sorted(groups)  # 03.31.2026 ADDED: save the group options for this experiment

    USE_LOADED_RESULTS = True  # 03.31.2026 CHANGED: later cells should browse the saved results instead of training again
    print("[LOADED DIR] OUTDIR:", OUTDIR)  # 03.31.2026 ADDED: confirm which saved batch directory is active
    print("[LOADED DIR] experiments:", SAVED_EXPERIMENT_NAMES)  # 03.31.2026 ADDED: show the saved experiment names discovered from disk

In [5]:
# Cell 3 — Channel groups + feature builder (uses derived/outs_fill)  # 03.31.2026 CHANGED: validate explicit experiment dictionaries against the available channel groups

CHANNEL_GROUPS = {  # 03.31.2026 CHANGED: available channel group definitions for every experiment
    "Classic4": ["l_on", "l_off", "m_on", "m_off"],  # 03.31.2026 CHANGED: classical L/M midget channels
    "Classic4_plus_s": ["l_on", "l_off", "m_on", "m_off", "s_on"],  # 03.31.2026 CHANGED: classical channels plus S-ON
    "Neitz4": ["l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off"],  # 03.31.2026 CHANGED: H2-linked midget channels only
    "Neitz8": ["l_on", "l_off", "m_on", "m_off", "l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off"],  # 03.31.2026 CHANGED: classical plus H2-linked channels
    "All9": ["l_on", "l_off", "m_on", "m_off", "l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off", "s_on"],  # 03.31.2026 CHANGED: all available channels together
}  # 03.31.2026 CHANGED: end channel-group definitions

VALID_MODEL_NAMES = {"RF", "MLR", "SVM"}  # 03.31.2026 ADDED: valid computation-model names for experiment validation
BUILD_GROUPS = []  # 03.31.2026 CHANGED: union of all groups requested across experiment dictionaries
EXP_NAMES = [str(_exp["name"]) for _exp in EXPERIMENTS]  # 03.31.2026 ADDED: ordered experiment names used to validate unique experiment folders
if len(set(EXP_NAMES)) != len(EXP_NAMES):  # 03.31.2026 ADDED: fail clearly when experiment names are duplicated
    raise ValueError("Each experiment name must be unique. Found duplicates in: {}".format(EXP_NAMES))  # 03.31.2026 ADDED: clear error for duplicate experiment names


for _exp in EXPERIMENTS:  # 03.31.2026 ADDED: validate every explicit experiment dictionary before running feature builds
    if "name" not in _exp or str(_exp["name"]).strip() == "":  # 03.31.2026 ADDED: require a non-empty experiment name
        raise ValueError("Each experiment must define a non-empty 'name'.")  # 03.31.2026 ADDED: clear error when an experiment name is missing

    if "groups" not in _exp or len(_exp["groups"]) == 0:  # 03.31.2026 ADDED: require at least one group in every experiment
        raise ValueError("Each experiment must define at least one channel group.")  # 03.31.2026 ADDED: clear error when groups are missing

    if "models" not in _exp or len(_exp["models"]) == 0:  # 03.31.2026 ADDED: require at least one model in every experiment
        raise ValueError("Each experiment must define at least one model.")  # 03.31.2026 ADDED: clear error when models are missing

    _exp["groups"] = [str(_g) for _g in _exp["groups"]]  # 03.31.2026 ADDED: normalize experiment group names into strings
    _exp["models"] = [str(_m).upper() for _m in _exp["models"]]  # 03.31.2026 ADDED: normalize experiment model names into uppercase strings
    _exp["seed"] = int(_exp["seed"])  # 03.31.2026 ADDED: normalize the experiment seed into an integer
    _exp["merge_gray_classes"] = bool(_exp["merge_gray_classes"])  # 03.31.2026 ADDED: normalize the gray-merge flag into a bool

    if _exp["subset_max_per_class"] is not None:  # 03.31.2026 ADDED: validate the optional per-class subset size
        _exp["subset_max_per_class"] = int(_exp["subset_max_per_class"])  # 03.31.2026 ADDED: normalize the subset size into an integer
        if _exp["subset_max_per_class"] <= 0:  # 03.31.2026 ADDED: require a positive subset size when one is provided
            raise ValueError("subset_max_per_class must be a positive integer or None.")  # 03.31.2026 ADDED: clear error for an invalid subset size

    unknown_groups = sorted(set(_exp["groups"]) - set(CHANNEL_GROUPS.keys()))  # 03.31.2026 ADDED: detect any experiment group names that are not defined above
    if len(unknown_groups) > 0:  # 03.31.2026 ADDED: fail clearly if an experiment asks for an unknown group
        raise ValueError("Unknown channel groups in experiment '{}': {}".format(_exp["name"], unknown_groups))  # 03.31.2026 ADDED: clear error for unknown groups

    unknown_models = sorted(set(_exp["models"]) - VALID_MODEL_NAMES)  # 03.31.2026 ADDED: detect any experiment model names that are not supported
    if len(unknown_models) > 0:  # 03.31.2026 ADDED: fail clearly if an experiment asks for an unknown model
        raise ValueError("Unknown models in experiment '{}': {}".format(_exp["name"], unknown_models))  # 03.31.2026 ADDED: clear error for unknown models

    BUILD_GROUPS.extend(_exp["groups"])  # 03.31.2026 ADDED: collect this experiment's groups into the overall requested group list

BUILD_GROUPS = sorted(set(BUILD_GROUPS))  # 03.31.2026 CHANGED: keep a de-duplicated sorted union of all requested groups across experiments

key_to_idx = {k: i for i, k in enumerate(keys_order)}  # 02.23.2026 CHANGED: map response-key name -> channel index in outs_fill_z

DOWNSAMPLE_MAPS = True  # 02.23.2026 CHANGED: keep response-map downsampling enabled by default
TARGET_HW = (256, 256)  # 02.23.2026 CHANGED: target response-map size used before flattening
DOWNSAMPLE_ORDER = 1  # 02.23.2026 CHANGED: bilinear interpolation order for response-map downsampling

if DOWNSAMPLE_MAPS:  # 02.23.2026 CHANGED: choose the feature-map height and width after downsampling
    H = int(TARGET_HW[0])  # 02.23.2026 CHANGED: downsampled response-map height
    W = int(TARGET_HW[1])  # 02.23.2026 CHANGED: downsampled response-map width
    print("[DOWNSAMPLE] {}x{} -> {}x{} (order={})".format(H0, W0, H, W, int(DOWNSAMPLE_ORDER)))  # 02.23.2026 CHANGED: show the response-map downsampling plan
else:  # 02.23.2026 CHANGED: keep original response-map size when downsampling is off
    H, W = H0, W0  # 02.23.2026 CHANGED: original response-map height and width

def get_map(stim_id, ch_idx):  # 02.23.2026 CHANGED: single-run response-map accessor
    A = np.asarray(outs_fill_z[int(stim_id), int(ch_idx), :, :], dtype=np.float32)  # 02.23.2026 CHANGED: read one response map from outs_fill
    if DOWNSAMPLE_MAPS:  # 02.23.2026 CHANGED: optionally downsample the response map before flattening
        z0 = float(H) / float(A.shape[0])  # 02.23.2026 CHANGED: vertical zoom factor for the response map
        z1 = float(W) / float(A.shape[1])  # 02.23.2026 CHANGED: horizontal zoom factor for the response map
        A = zoom(A, (z0, z1), order=int(DOWNSAMPLE_ORDER)).astype(np.float32, copy=False)  # 02.23.2026 CHANGED: downsample the response map into the configured target size
    return A  # 02.23.2026 CHANGED: return one response map ready for stacking and flattening

def build_X(keys_subset, stim_ids, mmap_path, desc):  # 02.23.2026 CHANGED: memmap feature builder shared by all experiments
    stim_ids = np.asarray(stim_ids, dtype=np.int64)  # 02.23.2026 CHANGED: normalize stimulus ids into an integer array
    ch_idxs = [int(key_to_idx[k]) for k in keys_subset]  # 02.23.2026 CHANGED: convert response-key names into channel indices

    n_feat = int(len(ch_idxs) * H * W)  # 02.23.2026 CHANGED: total flattened feature count for the selected channel set
    X = np.memmap(str(mmap_path), dtype=np.float32, mode="w+", shape=(len(stim_ids), n_feat))  # 02.23.2026 CHANGED: write features into an on-disk memmap array

    for row, sid in enumerate(tqdm(stim_ids, desc=desc)):  # 02.23.2026 CHANGED: build one flattened response vector per stimulus id
        maps = [get_map(int(sid), int(ch)) for ch in ch_idxs]  # 02.23.2026 CHANGED: load the selected response maps for this stimulus
        X[row] = np.stack(maps, axis=0).reshape(-1)  # 02.23.2026 CHANGED: stack and flatten the response maps into one feature vector

    X.flush()  # 02.23.2026 CHANGED: force the completed feature memmap to disk
    return X  # 02.23.2026 CHANGED: return the memmap feature matrix

print("Validated experiments for groups:", BUILD_GROUPS)  # 03.31.2026 ADDED: show the union of groups requested across the configured experiments

[DOWNSAMPLE] 1000x1000 -> 256x256 (order=1)


In [6]:
# Cell 4 — Targets (y) + experiment data helpers from metadata.csv  # 03.31.2026 CHANGED: prepare labels and optional balanced subsets per experiment

BASE_CLASS_ORDER = ["gray_d", "gray_l", "red", "green", "blue", "yellow"]  # 03.31.2026 ADDED: canonical unmerged class order for experiment preparation
BASE_TARGET_CLASS_WEIGHTS = {  # 03.31.2026 ADDED: class weights used by the split helper when gray classes stay separate
    "gray_l": 25.0,  # 03.31.2026 ADDED: keep the light-gray target weight from the prior notebook logic
    "gray_d": 25.0,  # 03.31.2026 ADDED: keep the dark-gray target weight from the prior notebook logic
    "red": 12.5,  # 03.31.2026 ADDED: keep the red target weight from the prior notebook logic
    "green": 12.5,  # 03.31.2026 ADDED: keep the green target weight from the prior notebook logic
    "blue": 12.5,  # 03.31.2026 ADDED: keep the blue target weight from the prior notebook logic
    "yellow": 12.5,  # 03.31.2026 ADDED: keep the yellow target weight from the prior notebook logic
}  # 03.31.2026 ADDED: end unmerged class-weight dictionary

MERGED_CLASS_ORDER = ["gray", "red", "green", "blue", "yellow"]  # 03.31.2026 ADDED: class order used when gray_d and gray_l are merged
MERGED_TARGET_CLASS_WEIGHTS = {  # 03.31.2026 ADDED: class weights used by the split helper when gray classes are merged
    "gray": 50.0,  # 03.31.2026 ADDED: combined gray weight equals gray_l plus gray_d
    "red": 12.5,  # 03.31.2026 ADDED: keep the red target weight from the prior notebook logic
    "green": 12.5,  # 03.31.2026 ADDED: keep the green target weight from the prior notebook logic
    "blue": 12.5,  # 03.31.2026 ADDED: keep the blue target weight from the prior notebook logic
    "yellow": 12.5,  # 03.31.2026 ADDED: keep the yellow target weight from the prior notebook logic
}  # 03.31.2026 ADDED: end merged class-weight dictionary

def prepare_experiment_data(meta_source, merge_gray_classes, subset_max_per_class, seed):  # 03.31.2026 ADDED: build experiment-specific labels, counts, and an optional balanced subset
    meta_exp = meta_source.copy()  # 03.31.2026 ADDED: keep experiment-specific label edits isolated from the source metadata
    label_series = meta_exp["true_label"].astype(str).copy()  # 03.31.2026 ADDED: start from the auto labels only

    if bool(merge_gray_classes):  # 03.31.2026 ADDED: optionally merge gray_d and gray_l into one gray class
        label_series = label_series.replace({"gray_d": "gray", "gray_l": "gray"})  # 03.31.2026 ADDED: collapse both gray classes into one shared label
        class_order = list(MERGED_CLASS_ORDER)  # 03.31.2026 ADDED: use the merged class order for this experiment
        target_class_weights = dict(MERGED_TARGET_CLASS_WEIGHTS)  # 03.31.2026 ADDED: use the merged class weights for this experiment
    else:  # 03.31.2026 ADDED: keep gray_d and gray_l separate when the merge switch is off
        class_order = list(BASE_CLASS_ORDER)  # 03.31.2026 ADDED: use the canonical unmerged class order for this experiment
        target_class_weights = dict(BASE_TARGET_CLASS_WEIGHTS)  # 03.31.2026 ADDED: use the canonical unmerged class weights for this experiment

    unknown = sorted(set(label_series.unique()) - set(class_order))  # 03.31.2026 ADDED: detect any labels that do not belong to the current experiment class order
    if len(unknown) > 0:  # 03.31.2026 ADDED: fail clearly if metadata contains an unexpected label
        raise ValueError("Unknown labels in true_label for this experiment: {}".format(unknown))  # 03.31.2026 ADDED: clear error for unexpected labels

    meta_exp["experiment_true_label"] = label_series  # 03.31.2026 ADDED: store the experiment-specific label string next to the source metadata
    meta_exp["_obj_int_r"] = np.round(meta_exp["obj_int"].astype(float), 6)  # 03.31.2026 ADDED: rounded object brightness used by the split helper
    meta_exp["_obj_sat_r"] = np.round(meta_exp["obj_sat"].astype(float), 6)  # 03.31.2026 ADDED: rounded object saturation used by the split helper

    if subset_max_per_class is not None:  # 03.31.2026 ADDED: optionally keep a smaller balanced subset per class
        rng = np.random.default_rng(int(seed))  # 03.31.2026 ADDED: reproducible subset selection for this experiment
        selected_parts = []  # 03.31.2026 ADDED: collect one shuffled subset per class

        for lab in class_order:  # 03.31.2026 ADDED: sample the same maximum count from every class that is present
            lab_ids = meta_exp.index[meta_exp["experiment_true_label"] == lab].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: stimulus ids in the current class
            rng.shuffle(lab_ids)  # 03.31.2026 ADDED: randomize the class-specific subset order reproducibly
            take_n = min(int(subset_max_per_class), int(lab_ids.size))  # 03.31.2026 ADDED: never request more ids than this class contains
            selected_parts.append(lab_ids[:take_n])  # 03.31.2026 ADDED: keep the balanced class-specific subset for this experiment

        selected_ids = np.sort(np.concatenate(selected_parts).astype(np.int64))  # 03.31.2026 ADDED: merge the class-specific subsets into one sorted stimulus-id array
        meta_exp = meta_exp.loc[selected_ids].copy()  # 03.31.2026 ADDED: restrict the experiment metadata to the chosen balanced subset
        label_series = meta_exp["experiment_true_label"].astype(str).copy()  # 03.31.2026 ADDED: refresh the label series after subsetting

    label_to_int = {lab: i for i, lab in enumerate(class_order)}  # 03.31.2026 ADDED: integer mapping for the experiment-specific class order
    y_series = pd.Series(label_series.map(label_to_int).to_numpy(dtype=np.int64), index=meta_exp.index, name="label_int")  # 03.31.2026 ADDED: label integers indexed by stim_ID for safe subset-aware lookup

    print("Class order:", class_order)  # 03.31.2026 ADDED: show the class order used by the current experiment
    print("Counts by class:", label_series.value_counts().reindex(class_order).fillna(0).astype(int).to_dict())  # 03.31.2026 ADDED: show class counts after optional merging and subsetting
    print("Counts by bg_hue:", meta_exp["bg_hue"].astype(str).value_counts().to_dict())  # 03.31.2026 ADDED: show background-hue counts after optional subsetting

    return {"meta": meta_exp, "label_series": label_series, "y_series": y_series, "class_order": class_order, "label_to_int": label_to_int, "target_class_weights": target_class_weights}  # 03.31.2026 ADDED: return all experiment-specific label objects needed downstream

Class order: ['gray_d', 'gray_l', 'red', 'green', 'blue', 'yellow']
Counts by class: {'gray_d': 925, 'gray_l': 925, 'red': 715, 'green': 715, 'blue': 715, 'yellow': 715}
Counts by bg_hue: {'red': 1040, 'green': 1040, 'blue': 1040, 'yellow': 1040, 'gray': 550}


In [7]:
# Cell 5 — Train/test split helpers (class weights + balance across obj_int/obj_sat + bg_hue)  # 03.31.2026 CHANGED: build and cache a split separately for each experiment

SPLIT_CACHE_DIR = RUN_DIR / "split_cache_v05"  # 03.31.2026 CHANGED: keep v05 split files separate from older notebook versions
SPLIT_CACHE_DIR.mkdir(exist_ok=True)  # 03.31.2026 CHANGED: create the v05 split-cache folder once

TEST_FRAC = 0.25  # 03.31.2026 CHANGED: held-out test fraction used for every experiment
N_INT_BINS = 3  # 03.31.2026 CHANGED: number of brightness bins used when balancing the split
N_SAT_BINS = 3  # 03.31.2026 CHANGED: number of saturation bins used when balancing the split
BALANCE_BG_TYPES = True  # 03.31.2026 CHANGED: keep background-hue balancing enabled for every experiment

def allocate_counts(total, labels, frac_dict):  # 03.31.2026 ADDED: convert target fractions into integer counts that sum to the requested total
    quotas = {lab: float(total) * float(frac_dict[lab]) for lab in labels}  # 03.31.2026 ADDED: ideal floating-point counts per label
    base = {lab: int(np.floor(quotas[lab])) for lab in labels}  # 03.31.2026 ADDED: floor the ideal counts before remainder allocation
    need = int(total - sum(base.values()))  # 03.31.2026 ADDED: number of counts still left to assign after flooring

    if need > 0:  # 03.31.2026 ADDED: allocate the remaining counts by descending fractional remainder
        rema = sorted(labels, key=lambda lab: quotas[lab] - base[lab], reverse=True)  # 03.31.2026 ADDED: labels ordered by how much fractional count they still deserve
        for lab in rema[:need]:  # 03.31.2026 ADDED: distribute the remaining counts to the highest remainders first
            base[lab] += 1  # 03.31.2026 ADDED: assign one additional count to this label

    return base  # 03.31.2026 ADDED: integer counts that sum to the requested total

def bin_continuous(values, n_bins):  # 03.31.2026 ADDED: convert one continuous metadata column into evenly spaced integer bins
    v = np.asarray(values, dtype=float)  # 03.31.2026 ADDED: normalize the continuous values into a float array
    edges = np.linspace(float(np.nanmin(v)), float(np.nanmax(v)), int(n_bins) + 1)  # 03.31.2026 ADDED: evenly spaced bin edges across the observed range
    mids = edges[1:-1]  # 03.31.2026 ADDED: inner edges become the digitize boundaries
    out = np.digitize(v, mids, right=False).astype(np.int64)  # 03.31.2026 ADDED: assign each value to one bin index
    return np.clip(out, 0, int(n_bins) - 1)  # 03.31.2026 ADDED: keep all bin ids inside the valid range

def sample_balanced(idxs, n_select, rng, int_bin_series, sat_bin_series, bg_id_series=None):  # 03.31.2026 ADDED: sample indices while balancing brightness, saturation, and optional background hue
    idxs = np.asarray(idxs, dtype=np.int64)  # 03.31.2026 ADDED: normalize candidate indices into an integer array

    if n_select <= 0:  # 03.31.2026 ADDED: handle an empty selection request clearly
        return np.array([], dtype=np.int64)  # 03.31.2026 ADDED: return an empty integer array when no samples are requested

    if n_select >= len(idxs):  # 03.31.2026 ADDED: return a shuffled copy when the request equals or exceeds the available ids
        out = idxs.copy()  # 03.31.2026 ADDED: copy the full candidate set before shuffling
        rng.shuffle(out)  # 03.31.2026 ADDED: shuffle the full candidate set reproducibly
        return out  # 03.31.2026 ADDED: return all available ids in randomized order

    cells = {}  # 03.31.2026 ADDED: map each balancing-cell key to the stimulus ids that belong to it

    for i in idxs:  # 03.31.2026 ADDED: build one balancing cell per candidate stimulus id
        if bg_id_series is None:  # 03.31.2026 ADDED: use brightness and saturation only when background balancing is off
            key = (int(int_bin_series.loc[int(i)]), int(sat_bin_series.loc[int(i)]))  # 03.31.2026 ADDED: balancing-cell key without background hue
        else:  # 03.31.2026 ADDED: include background hue when background balancing is on
            key = (int(int_bin_series.loc[int(i)]), int(sat_bin_series.loc[int(i)]), int(bg_id_series.loc[int(i)]))  # 03.31.2026 ADDED: balancing-cell key with background hue

        cells.setdefault(key, []).append(int(i))  # 03.31.2026 ADDED: append this stimulus id to its balancing cell

    cell_keys = list(cells.keys())  # 03.31.2026 ADDED: list of balancing-cell keys present in the candidate set
    rng.shuffle(cell_keys)  # 03.31.2026 ADDED: randomize the cell order before quota allocation

    base = {ck: (n_select // len(cell_keys)) for ck in cell_keys}  # 03.31.2026 ADDED: start with an equal base quota per balancing cell
    rem = int(n_select - sum(base.values()))  # 03.31.2026 ADDED: leftover samples to distribute after equal base allocation

    while rem > 0:  # 03.31.2026 ADDED: round-robin the remainder across cells with remaining capacity
        progressed = False  # 03.31.2026 ADDED: track whether any cell accepted another sample during this pass

        for ck in cell_keys:  # 03.31.2026 ADDED: visit each balancing cell in the randomized order
            if rem <= 0:  # 03.31.2026 ADDED: stop early when all remainder samples are assigned
                break  # 03.31.2026 ADDED: leave the round-robin loop once no remainder is left

            if base[ck] < len(cells[ck]):  # 03.31.2026 ADDED: only increase the quota for cells that still have unused ids
                base[ck] += 1  # 03.31.2026 ADDED: assign one more sample to this balancing cell
                rem -= 1  # 03.31.2026 ADDED: decrement the remainder after assigning one sample
                progressed = True  # 03.31.2026 ADDED: record that this round-robin pass made progress

        if not progressed:  # 03.31.2026 ADDED: stop when every balancing cell has reached capacity
            break  # 03.31.2026 ADDED: no further balanced allocation is possible

    selected = []  # 03.31.2026 ADDED: collect the ids chosen from each balancing cell

    for ck in cell_keys:  # 03.31.2026 ADDED: draw the allocated number of ids from every balancing cell
        arr = np.array(cells[ck], dtype=np.int64)  # 03.31.2026 ADDED: candidate ids for this balancing cell
        rng.shuffle(arr)  # 03.31.2026 ADDED: randomize the order inside the balancing cell
        take = int(base[ck])  # 03.31.2026 ADDED: number of ids assigned to this balancing cell
        if take > 0:  # 03.31.2026 ADDED: keep only non-empty selections
            selected.append(arr[:take])  # 03.31.2026 ADDED: save the chosen ids from this balancing cell

    if len(selected) > 0:  # 03.31.2026 ADDED: concatenate the chosen ids when at least one cell contributed ids
        selected = np.concatenate(selected).astype(np.int64)  # 03.31.2026 ADDED: flatten all cell-specific selections into one id array
    else:  # 03.31.2026 ADDED: keep an empty integer array when no ids were selected
        selected = np.array([], dtype=np.int64)  # 03.31.2026 ADDED: empty selection fallback

    if len(selected) < n_select:  # 03.31.2026 ADDED: top off the selection if balanced quotas could not fill the full request
        remaining = np.setdiff1d(idxs, selected, assume_unique=False).astype(np.int64)  # 03.31.2026 ADDED: ids still available after the balanced draw
        rng.shuffle(remaining)  # 03.31.2026 ADDED: randomize the remaining ids before topping off
        need = int(n_select - len(selected))  # 03.31.2026 ADDED: number of extra ids still needed
        selected = np.concatenate([selected, remaining[:need]]).astype(np.int64)  # 03.31.2026 ADDED: top off to the requested selection size

    rng.shuffle(selected)  # 03.31.2026 ADDED: randomize the final selected ids once more before returning
    return selected  # 03.31.2026 ADDED: balanced sampled ids for the requested split partition

def build_experiment_split(exp_name, meta_exp, y_series, class_order, target_class_weights, seed, split_suffix):  # 03.31.2026 ADDED: build or reload the train/test split for one experiment
    labels = list(class_order)  # 03.31.2026 ADDED: stable class-order copy for split reporting
    split_cache_path = SPLIT_CACHE_DIR / "{}_{}.npz".format(str(exp_name), str(split_suffix))  # 03.31.2026 ADDED: experiment-specific split-cache path

    if USE_SAVED_SPLIT and split_cache_path.exists():  # 03.31.2026 CHANGED: reuse the saved split when an identical experiment split file already exists
        with np.load(str(split_cache_path)) as split_npz:  # 03.31.2026 CHANGED: open the cached split file for this experiment
            train_idx = split_npz["train_idx"].astype(np.int64)  # 03.31.2026 CHANGED: reload the saved train stimulus ids
            test_idx = split_npz["test_idx"].astype(np.int64)  # 03.31.2026 CHANGED: reload the saved test stimulus ids

        print("Loaded saved split from:", split_cache_path)  # 03.31.2026 CHANGED: confirm split reuse for this experiment
        print("Split sizes:", {"train_pool": len(train_idx), "test": len(test_idx)})  # 03.31.2026 CHANGED: show the loaded split sizes
        print("Train counts:", meta_exp.loc[train_idx, "experiment_true_label"].value_counts().reindex(labels).fillna(0).astype(int).to_dict())  # 03.31.2026 CHANGED: show class counts in the loaded train split
        print("Test  counts:", meta_exp.loc[test_idx, "experiment_true_label"].value_counts().reindex(labels).fillna(0).astype(int).to_dict())  # 03.31.2026 CHANGED: show class counts in the loaded test split

        return train_idx, test_idx, split_cache_path  # 03.31.2026 ADDED: return the cached split and its path

    rng = np.random.default_rng(int(seed))  # 03.31.2026 CHANGED: experiment-specific random generator for a new split
    target_total = float(sum(target_class_weights.values()))  # 03.31.2026 ADDED: total weight across classes for this experiment
    target_class_frac = {k: float(v) / target_total for k, v in target_class_weights.items()}  # 03.31.2026 ADDED: class fractions implied by the experiment weight dictionary

    if BALANCE_BG_TYPES and ("bg_hue" not in meta_exp.columns):  # 03.31.2026 CHANGED: preserve the explicit metadata requirement for background balancing
        raise KeyError("BALANCE_BG_TYPES=True requires metadata column 'bg_hue'.")  # 03.31.2026 CHANGED: clear error when bg_hue is missing

    all_idx = meta_exp.index.to_numpy(dtype=np.int64)  # 03.31.2026 CHANGED: all stimulus ids available inside this experiment after optional subsetting
    idx_by_label = {lab: meta_exp.index[meta_exp["experiment_true_label"] == lab].to_numpy(dtype=np.int64) for lab in labels}  # 03.31.2026 CHANGED: stimulus ids grouped by experiment-specific label

    avail_by_label = {lab: int(len(idx_by_label[lab])) for lab in labels}  # 03.31.2026 ADDED: available sample count per class for this experiment
    total_eff_float = min(float(avail_by_label[lab]) / float(target_class_frac[lab]) for lab in labels)  # 03.31.2026 ADDED: largest feasible total sample count under the requested class fractions
    total_eff = int(np.floor(total_eff_float))  # 03.31.2026 ADDED: integer feasible total sample count
    total_eff = min(total_eff, int(len(all_idx)))  # 03.31.2026 ADDED: never exceed the number of available experiment samples

    test_total = int(round(TEST_FRAC * total_eff))  # 03.31.2026 ADDED: requested test size from the feasible total sample count
    test_total = max(test_total, len(labels))  # 03.31.2026 ADDED: keep at least one test sample available per class
    test_total = min(test_total, total_eff - len(labels))  # 03.31.2026 ADDED: keep at least one train sample available per class
    train_total = total_eff - test_total  # 03.31.2026 ADDED: remaining samples become the train pool size

    total_counts = allocate_counts(total_eff, labels, target_class_frac)  # 03.31.2026 ADDED: integer class counts for the feasible total sample set
    frac_eff = {lab: float(total_counts[lab]) / float(total_eff) for lab in labels}  # 03.31.2026 ADDED: effective class fractions after integer allocation
    test_counts = allocate_counts(test_total, labels, frac_eff)  # 03.31.2026 ADDED: integer class counts for the test split
    train_counts = {lab: int(total_counts[lab] - test_counts[lab]) for lab in labels}  # 03.31.2026 ADDED: integer class counts for the train split

    int_bin_series = pd.Series(bin_continuous(meta_exp["_obj_int_r"].to_numpy(), N_INT_BINS), index=meta_exp.index)  # 03.31.2026 ADDED: brightness bins indexed by stim_ID for this experiment
    sat_bin_series = pd.Series(bin_continuous(meta_exp["_obj_sat_r"].to_numpy(), N_SAT_BINS), index=meta_exp.index)  # 03.31.2026 ADDED: saturation bins indexed by stim_ID for this experiment

    if BALANCE_BG_TYPES:  # 03.31.2026 ADDED: compute one integer background id per stimulus when background balancing is on
        bg_str = meta_exp["bg_hue"].astype(str).to_numpy()  # 03.31.2026 ADDED: background hue as strings for this experiment
        bg_levels = sorted(np.unique(bg_str).tolist())  # 03.31.2026 ADDED: sorted unique background hues in this experiment
        bg_map = {b: i for i, b in enumerate(bg_levels)}  # 03.31.2026 ADDED: background hue -> integer id lookup
        bg_id_series = pd.Series(np.array([bg_map[b] for b in bg_str], dtype=np.int64), index=meta_exp.index)  # 03.31.2026 ADDED: background hue ids indexed by stim_ID for this experiment
    else:  # 03.31.2026 ADDED: keep a None marker when background balancing is off
        bg_id_series = None  # 03.31.2026 ADDED: background-balancing off marker

    test_parts = []  # 03.31.2026 ADDED: collect class-specific test ids
    train_parts = []  # 03.31.2026 ADDED: collect class-specific train ids

    for lab in labels:  # 03.31.2026 ADDED: sample one balanced train and test subset per class
        idxs = idx_by_label[lab]  # 03.31.2026 ADDED: candidate stimulus ids for the current class
        n_te = int(test_counts[lab])  # 03.31.2026 ADDED: requested number of test ids for the current class
        n_tr = int(train_counts[lab])  # 03.31.2026 ADDED: requested number of train ids for the current class
        need = n_te + n_tr  # 03.31.2026 ADDED: total ids required from the current class

        if need > len(idxs):  # 03.31.2026 ADDED: fail clearly when a class lacks enough samples for the requested split
            raise ValueError("Not enough '{}' samples: need {}, have {}".format(lab, need, len(idxs)))  # 03.31.2026 ADDED: clear error for an under-sized class

        te = sample_balanced(idxs, n_te, rng, int_bin_series, sat_bin_series, bg_id_series=bg_id_series)  # 03.31.2026 ADDED: balanced test ids for the current class
        rem = np.setdiff1d(idxs, te, assume_unique=False).astype(np.int64)  # 03.31.2026 ADDED: remaining ids after reserving the class-specific test ids
        tr = sample_balanced(rem, n_tr, rng, int_bin_series, sat_bin_series, bg_id_series=bg_id_series)  # 03.31.2026 ADDED: balanced train ids for the current class

        test_parts.append(te)  # 03.31.2026 ADDED: save the class-specific test ids
        train_parts.append(tr)  # 03.31.2026 ADDED: save the class-specific train ids

    test_idx = np.concatenate(test_parts).astype(np.int64)  # 03.31.2026 ADDED: final test stimulus ids for this experiment
    train_idx = np.concatenate(train_parts).astype(np.int64)  # 03.31.2026 ADDED: final train stimulus ids for this experiment

    rng.shuffle(test_idx)  # 03.31.2026 ADDED: shuffle the final test ids reproducibly
    rng.shuffle(train_idx)  # 03.31.2026 ADDED: shuffle the final train ids reproducibly

    print("Split sizes:", {"train_pool": len(train_idx), "test": len(test_idx)})  # 03.31.2026 ADDED: show the new split sizes
    print("Train counts:", meta_exp.loc[train_idx, "experiment_true_label"].value_counts().reindex(labels).fillna(0).astype(int).to_dict())  # 03.31.2026 ADDED: show class counts in the new train split
    print("Test  counts:", meta_exp.loc[test_idx, "experiment_true_label"].value_counts().reindex(labels).fillna(0).astype(int).to_dict())  # 03.31.2026 ADDED: show class counts in the new test split

    np.savez(str(split_cache_path), train_idx=np.asarray(train_idx, dtype=np.int64), test_idx=np.asarray(test_idx, dtype=np.int64))  # 03.31.2026 CHANGED: save the split immediately after creating it
    print("Saved split to:", split_cache_path)  # 03.31.2026 CHANGED: show where the experiment-specific reusable split file lives

    return train_idx, test_idx, split_cache_path  # 03.31.2026 ADDED: return the new split and its cache path

Loaded saved split from: export_retina_20260313_230715/split_cache/train_test_split_seed12.npz
Split sizes: {'train_pool': 2775, 'test': 925}
Train counts: {'gray_d': 694, 'gray_l': 694, 'red': 347, 'green': 347, 'blue': 346, 'yellow': 347}
Test  counts: {'gray_d': 231, 'gray_l': 231, 'red': 116, 'green': 116, 'blue': 116, 'yellow': 115}


In [ ]:
# Cell 6 — Cross-validated training + held-out test (per experiment, per channel group)  # 03.31.2026 CHANGED: loop through explicit experiments and save core outputs immediately

if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: skip new training when the notebook is browsing a saved results folder
    print("[USE_LOADED_RESULTS] Skipping training (Cell 6).")  # 03.31.2026 CHANGED: clear status message for saved-results mode
else:  # 03.31.2026 CHANGED: run every configured experiment now
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # 03.31.2026 CHANGED: one timestamped batch folder for this notebook run
    OUTDIR = Path("svm_mlr_rf_eval_batch_" + ts)  # 03.31.2026 CHANGED: save all experiment folders inside one batch root
    OUTDIR.mkdir(exist_ok=True)  # 03.31.2026 CHANGED: create the batch root once at the start of the notebook run
    CACHE_DIR = Path("rf_feature_cache_v05")  # 03.31.2026 CHANGED: keep v05 feature memmaps separate from older notebook versions
    CACHE_DIR.mkdir(exist_ok=True)  # 03.31.2026 CHANGED: create the shared v05 feature-cache folder once
    EXPERIMENT_RESULTS = []  # 03.31.2026 ADDED: collect in-memory results for every experiment for later cells
    EXPERIMENT_RESULTS_BY_NAME = {}  # 03.31.2026 ADDED: quick experiment-name lookup for later viewer and save cells

    def fit_mlr_on_Xy(X_train, y_train, seed):  # 03.31.2026 CHANGED: keep the original MLR training logic but accept an experiment seed
        mlr = make_pipeline(  # 03.31.2026 CHANGED: keep the scaler plus multinomial logistic regression pipeline
            StandardScaler(with_mean=False, copy=False),  # 03.31.2026 CHANGED: preserve sparse-friendly scaling before MLR
            LogisticRegression(solver="saga", C=float(MLR_C), max_iter=int(MLR_MAX_ITER), tol=float(MLR_TOL), random_state=int(seed)),  # 03.31.2026 CHANGED: keep the original MLR hyperparameters while using the experiment seed
        )  # 03.31.2026 CHANGED: end MLR pipeline definition
        mlr.fit(X_train, y_train)  # 03.31.2026 CHANGED: fit MLR on the provided features and labels
        return mlr  # 03.31.2026 CHANGED: return the fitted MLR pipeline

    def fit_svm_on_Xy(X_train, y_train):  # 03.31.2026 CHANGED: keep the original SVM training logic in one helper
        svm = make_pipeline(  # 03.31.2026 CHANGED: keep the scaler plus SVM pipeline
            StandardScaler(with_mean=False, copy=False),  # 03.31.2026 CHANGED: preserve sparse-friendly scaling before SVM
            SVC(kernel=str(SVM_KERNEL), C=float(SVM_C), gamma=SVM_GAMMA, probability=bool(SVM_PROBABILITY)),  # 03.31.2026 CHANGED: keep the shared SVM hyperparameters unchanged
        )  # 03.31.2026 CHANGED: end SVM pipeline definition
        svm.fit(X_train, y_train)  # 03.31.2026 CHANGED: fit SVM on the provided features and labels
        return svm  # 03.31.2026 CHANGED: return the fitted SVM pipeline

    for exp_idx, exp_cfg in enumerate(EXPERIMENTS, start=1):  # 03.31.2026 ADDED: run one full train/test workflow per experiment dictionary
        exp_name = str(exp_cfg["name"])  # 03.31.2026 ADDED: stable name for this experiment folder and summary rows
        exp_groups = list(exp_cfg["groups"])  # 03.31.2026 ADDED: channel groups requested for this experiment
        exp_models = [str(_m).upper() for _m in exp_cfg["models"]]  # 03.31.2026 ADDED: model names requested for this experiment
        exp_seed = int(exp_cfg["seed"])  # 03.31.2026 ADDED: seed used for this experiment's split and model fits
        exp_merge_gray_classes = bool(exp_cfg["merge_gray_classes"])  # 03.31.2026 ADDED: whether this experiment merges gray_d and gray_l
        exp_subset_max_per_class = exp_cfg["subset_max_per_class"]  # 03.31.2026 ADDED: optional balanced subset size per class for this experiment
        exp_subset_label = "full" if exp_subset_max_per_class is None else "subset{}".format(int(exp_subset_max_per_class))  # 03.31.2026 ADDED: short subset tag for cache and save filenames
        exp_split_suffix = "seed{}_merge{}_{}".format(int(exp_seed), int(exp_merge_gray_classes), str(exp_subset_label))  # 03.31.2026 ADDED: unique split-cache suffix for this experiment configuration
        exp_outdir = OUTDIR / exp_name  # 03.31.2026 ADDED: folder that holds all saved artifacts for this experiment
        exp_outdir.mkdir(parents=True, exist_ok=True)  # 03.31.2026 ADDED: create the experiment folder before saving any artifact
        exp_modeldir = exp_outdir / "models"  # 03.31.2026 ADDED: dedicated model folder for this experiment
        exp_modeldir.mkdir(exist_ok=True)  # 03.31.2026 ADDED: create the experiment model folder now
        exp_prediction_dir = exp_outdir / "prediction_cache"  # 03.31.2026 ADDED: save prediction bundles immediately as they are generated
        exp_prediction_dir.mkdir(exist_ok=True)  # 03.31.2026 ADDED: create the prediction-cache folder now
        exp_split_path = exp_outdir / "split_indices.npz"  # 03.31.2026 ADDED: experiment-local copy of the exact train/test ids
        exp_context_path = exp_outdir / "run_context.json"  # 03.31.2026 ADDED: experiment-local context file for later browsing
        exp_saved_model_rows = []  # 03.31.2026 ADDED: manifest rows for the models saved during this experiment

        print("\n===== Experiment {}/{}: {} =====".format(int(exp_idx), int(N_EXPERIMENTS), exp_name))  # 03.31.2026 ADDED: clear experiment header in the notebook output
        exp_data = prepare_experiment_data(meta, exp_merge_gray_classes, exp_subset_max_per_class, exp_seed)  # 03.31.2026 ADDED: build labels and an optional balanced subset for this experiment
        exp_meta = exp_data["meta"]  # 03.31.2026 ADDED: experiment-specific metadata after optional label merging and subsetting
        exp_y_series = exp_data["y_series"]  # 03.31.2026 ADDED: experiment label integers indexed by stim_ID
        exp_class_order = exp_data["class_order"]  # 03.31.2026 ADDED: experiment-specific class order used for training and evaluation
        exp_target_class_weights = exp_data["target_class_weights"]  # 03.31.2026 ADDED: experiment-specific class weights for split balancing
        exp_train_idx, exp_test_idx, exp_split_cache_path = build_experiment_split(exp_name, exp_meta, exp_y_series, exp_class_order, exp_target_class_weights, exp_seed, exp_split_suffix)  # 03.31.2026 ADDED: build or reload the exact split for this experiment
        np.savez(str(exp_split_path), train_idx=np.asarray(exp_train_idx, dtype=np.int64), test_idx=np.asarray(exp_test_idx, dtype=np.int64))  # 03.31.2026 ADDED: save an experiment-local copy of the split immediately
        print("Saved experiment split copy to:", exp_split_path)  # 03.31.2026 ADDED: show where the experiment-local split copy lives

        cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=int(exp_seed))  # 03.31.2026 CHANGED: keep the original 2-fold CV logic while using the experiment seed
        rf_models = {}  # 03.31.2026 ADDED: fitted RF models keyed by group for this experiment
        mlr_models = {}  # 03.31.2026 ADDED: fitted MLR models keyed by group for this experiment
        svm_models = {}  # 03.31.2026 ADDED: fitted SVM models keyed by group for this experiment
        cv_scores_by_group = {}  # 03.31.2026 ADDED: RF CV scores keyed by group for this experiment
        cv_scores_mlr_by_group = {}  # 03.31.2026 ADDED: MLR CV scores keyed by group for this experiment
        cv_scores_svm_by_group = {}  # 03.31.2026 ADDED: SVM CV scores keyed by group for this experiment
        test_scores_by_group = {}  # 03.31.2026 ADDED: RF held-out test scores keyed by group for this experiment
        test_scores_mlr_by_group = {}  # 03.31.2026 ADDED: MLR held-out test scores keyed by group for this experiment
        test_scores_svm_by_group = {}  # 03.31.2026 ADDED: SVM held-out test scores keyed by group for this experiment
        y_test_pred_by_group = {}  # 03.31.2026 ADDED: RF held-out predictions keyed by group for this experiment
        y_test_pred_mlr_by_group = {}  # 03.31.2026 ADDED: MLR held-out predictions keyed by group for this experiment
        y_test_pred_svm_by_group = {}  # 03.31.2026 ADDED: SVM held-out predictions keyed by group for this experiment
        y_train_pool = exp_y_series.loc[exp_train_idx].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: train-pool labels aligned to exp_train_idx order

        for gname in exp_groups:  # 03.31.2026 CHANGED: fit the requested models for every group in this experiment
            keys_subset = CHANNEL_GROUPS[gname]  # 03.31.2026 CHANGED: channel keys used to build features for this group
            fold_scores_rf = []  # 03.31.2026 ADDED: RF CV scores for this group
            fold_scores_mlr = []  # 03.31.2026 ADDED: MLR CV scores for this group
            fold_scores_svm = []  # 03.31.2026 ADDED: SVM CV scores for this group

            print("\n=== {} / {} (channels={}) ===".format(exp_name, gname, len(keys_subset)))  # 03.31.2026 CHANGED: clearer group header that includes the experiment name

            for fold, (tr_rel, va_rel) in enumerate(cv.split(np.zeros(len(exp_train_idx)), y_train_pool)):  # 03.31.2026 CHANGED: preserve the original CV loop on this experiment's train pool
                tr_global = exp_train_idx[tr_rel]  # 03.31.2026 CHANGED: training stim_IDs for this fold
                va_global = exp_train_idx[va_rel]  # 03.31.2026 CHANGED: validation stim_IDs for this fold
                xtr_path = CACHE_DIR / "{}_{}_fold{}_train.dat".format(exp_name, gname, int(fold))  # 03.31.2026 CHANGED: experiment-specific train memmap path for this fold
                xva_path = CACHE_DIR / "{}_{}_fold{}_val.dat".format(exp_name, gname, int(fold))  # 03.31.2026 CHANGED: experiment-specific validation memmap path for this fold
                X_tr = build_X(keys_subset, tr_global, xtr_path, desc="{} {} fold{} X_train".format(exp_name, gname, int(fold)))  # 03.31.2026 CHANGED: build fold train features for this experiment and group
                X_va = build_X(keys_subset, va_global, xva_path, desc="{} {} fold{} X_val".format(exp_name, gname, int(fold)))  # 03.31.2026 CHANGED: build fold validation features for this experiment and group
                msg = ["  fold {}:".format(int(fold))]  # 03.31.2026 ADDED: one concise score line per fold

                if "RF" in exp_models:  # 03.31.2026 CHANGED: fit RF only when this experiment requested RF
                    rf = RandomForestClassifier(n_estimators=int(N_EST), max_depth=int(MAX_DEPTH), random_state=int(exp_seed), n_jobs=int(RF_N_JOBS))  # 03.31.2026 CHANGED: keep the original RF hyperparameters while using the experiment seed
                    rf.fit(X_tr, exp_y_series.loc[tr_global].to_numpy(dtype=np.int64))  # 03.31.2026 CHANGED: fit RF on the current fold train split
                    acc = float(rf.score(X_va, exp_y_series.loc[va_global].to_numpy(dtype=np.int64)))  # 03.31.2026 CHANGED: score RF on the current fold validation split
                    fold_scores_rf.append(acc)  # 03.31.2026 CHANGED: save the RF fold score for this group
                    msg.append("RF={:.4f}".format(acc))  # 03.31.2026 CHANGED: print the RF fold score inline
                    del rf  # 03.31.2026 CHANGED: free the temporary RF fold model promptly

                if "MLR" in exp_models:  # 03.31.2026 CHANGED: fit MLR only when this experiment requested MLR
                    mlr_cv = fit_mlr_on_Xy(X_tr, exp_y_series.loc[tr_global].to_numpy(dtype=np.int64), exp_seed)  # 03.31.2026 CHANGED: fit MLR on the current fold train split
                    acc_mlr = float(mlr_cv.score(X_va, exp_y_series.loc[va_global].to_numpy(dtype=np.int64)))  # 03.31.2026 CHANGED: score MLR on the current fold validation split
                    fold_scores_mlr.append(acc_mlr)  # 03.31.2026 CHANGED: save the MLR fold score for this group
                    msg.append("MLR={:.4f}".format(acc_mlr))  # 03.31.2026 CHANGED: print the MLR fold score inline
                    del mlr_cv  # 03.31.2026 CHANGED: free the temporary MLR fold model promptly

                if "SVM" in exp_models:  # 03.31.2026 CHANGED: fit SVM only when this experiment requested SVM
                    svm_cv = fit_svm_on_Xy(X_tr, exp_y_series.loc[tr_global].to_numpy(dtype=np.int64))  # 03.31.2026 CHANGED: fit SVM on the current fold train split
                    acc_svm = float(svm_cv.score(X_va, exp_y_series.loc[va_global].to_numpy(dtype=np.int64)))  # 03.31.2026 CHANGED: score SVM on the current fold validation split
                    fold_scores_svm.append(acc_svm)  # 03.31.2026 CHANGED: save the SVM fold score for this group
                    msg.append("SVM={:.4f}".format(acc_svm))  # 03.31.2026 CHANGED: print the SVM fold score inline
                    del svm_cv  # 03.31.2026 CHANGED: free the temporary SVM fold model promptly

                print(" ".join(msg))  # 03.31.2026 CHANGED: display the fold scores for the current group
                del X_tr, X_va  # 03.31.2026 CHANGED: release the fold memmap handles before deleting their files
                gc.collect()  # 03.31.2026 CHANGED: encourage cleanup between fold feature builds
                if Path(xtr_path).exists():  # 03.31.2026 CHANGED: guard train memmap deletion in case the file is already gone
                    Path(xtr_path).unlink()  # 03.31.2026 CHANGED: delete the fold train memmap file now
                if Path(xva_path).exists():  # 03.31.2026 CHANGED: guard validation memmap deletion in case the file is already gone
                    Path(xva_path).unlink()  # 03.31.2026 CHANGED: delete the fold validation memmap file now

            if "RF" in exp_models:  # 03.31.2026 CHANGED: save RF CV scores for this group when RF ran
                cv_scores_by_group[gname] = fold_scores_rf  # 03.31.2026 CHANGED: RF CV scores keyed by group
                print("CV RF  mean±std: {:.4f} ± {:.4f}".format(float(np.mean(fold_scores_rf)), float(np.std(fold_scores_rf))))  # 03.31.2026 CHANGED: report the RF CV summary for this group
            if "MLR" in exp_models:  # 03.31.2026 CHANGED: save MLR CV scores for this group when MLR ran
                cv_scores_mlr_by_group[gname] = fold_scores_mlr  # 03.31.2026 CHANGED: MLR CV scores keyed by group
                print("CV MLR mean±std: {:.4f} ± {:.4f}".format(float(np.mean(fold_scores_mlr)), float(np.std(fold_scores_mlr))))  # 03.31.2026 CHANGED: report the MLR CV summary for this group
            if "SVM" in exp_models:  # 03.31.2026 CHANGED: save SVM CV scores for this group when SVM ran
                cv_scores_svm_by_group[gname] = fold_scores_svm  # 03.31.2026 CHANGED: SVM CV scores keyed by group
                print("CV SVM mean±std: {:.4f} ± {:.4f}".format(float(np.mean(fold_scores_svm)), float(np.std(fold_scores_svm))))  # 03.31.2026 CHANGED: report the SVM CV summary for this group

            xtrain_path = CACHE_DIR / "{}_{}_train_full.dat".format(exp_name, gname)  # 03.31.2026 CHANGED: experiment-specific full-train memmap path for this group
            X_train_full = build_X(keys_subset, exp_train_idx, xtrain_path, desc="{} {} X_train_full".format(exp_name, gname))  # 03.31.2026 CHANGED: build the full train features for this experiment and group

            if "RF" in exp_models:  # 03.31.2026 CHANGED: fit the final RF model for this group when RF ran
                rf_final = RandomForestClassifier(n_estimators=int(N_EST), max_depth=int(MAX_DEPTH), random_state=int(exp_seed), n_jobs=int(RF_N_JOBS))  # 03.31.2026 CHANGED: final RF model with the shared RF hyperparameters
                rf_final.fit(X_train_full, exp_y_series.loc[exp_train_idx].to_numpy(dtype=np.int64))  # 03.31.2026 CHANGED: fit RF on the full experiment train pool
            if "MLR" in exp_models:  # 03.31.2026 CHANGED: fit the final MLR model for this group when MLR ran
                mlr_final = fit_mlr_on_Xy(X_train_full, exp_y_series.loc[exp_train_idx].to_numpy(dtype=np.int64), exp_seed)  # 03.31.2026 CHANGED: fit MLR on the full experiment train pool
            if "SVM" in exp_models:  # 03.31.2026 CHANGED: fit the final SVM model for this group when SVM ran
                svm_final = fit_svm_on_Xy(X_train_full, exp_y_series.loc[exp_train_idx].to_numpy(dtype=np.int64))  # 03.31.2026 CHANGED: fit SVM on the full experiment train pool

            del X_train_full  # 03.31.2026 CHANGED: release the full-train memmap handle before deleting its file
            gc.collect()  # 03.31.2026 CHANGED: encourage cleanup before the test feature build
            if Path(xtrain_path).exists():  # 03.31.2026 CHANGED: guard full-train memmap deletion in case the file is already gone
                Path(xtrain_path).unlink()  # 03.31.2026 CHANGED: delete the full-train memmap file now

            xtest_path = CACHE_DIR / "{}_{}_test_full.dat".format(exp_name, gname)  # 03.31.2026 CHANGED: experiment-specific full-test memmap path for this group
            X_test_full = build_X(keys_subset, exp_test_idx, xtest_path, desc="{} {} X_test_full".format(exp_name, gname))  # 03.31.2026 CHANGED: build the full test features for this experiment and group
            y_true_test = exp_y_series.loc[exp_test_idx].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: held-out test labels aligned to exp_test_idx order for this experiment

            if "RF" in exp_models:  # 03.31.2026 CHANGED: generate and save RF predictions immediately when RF ran
                y_pred_test = np.asarray(rf_final.predict(X_test_full), dtype=np.int64)  # 03.31.2026 CHANGED: RF held-out predictions for this group
                test_acc = float(np.mean(y_pred_test == y_true_test))  # 03.31.2026 CHANGED: RF held-out accuracy for this group
                rf_models[gname] = rf_final  # 03.31.2026 CHANGED: keep the fitted RF model in memory for later cells
                y_test_pred_by_group[gname] = y_pred_test  # 03.31.2026 CHANGED: keep RF held-out predictions in memory for later cells
                test_scores_by_group[gname] = test_acc  # 03.31.2026 CHANGED: keep RF held-out accuracy in memory for later cells
                rf_pred_path = exp_prediction_dir / "test_predictions_RF_{}.npz".format(gname)  # 03.31.2026 ADDED: experiment-local RF prediction bundle path
                np.savez(str(rf_pred_path), test_idx=np.asarray(exp_test_idx, dtype=np.int64), y_true=np.asarray(y_true_test, dtype=np.int64), y_pred=np.asarray(y_pred_test, dtype=np.int64), class_order=np.asarray(exp_class_order, dtype=object))  # 03.31.2026 ADDED: save RF predictions immediately after they are generated
                rf_model_path = exp_modeldir / "RF_model_{}.joblib".format(gname)  # 03.31.2026 ADDED: experiment-local RF model path
                joblib.dump(rf_final, rf_model_path)  # 03.31.2026 ADDED: save the fitted RF model immediately after it is generated
                exp_saved_model_rows.append({"model": "RF", "group": gname, "path": str(rf_model_path)})  # 03.31.2026 ADDED: record the saved RF model in the experiment manifest
                print("FINAL test acc (RF) : {:.4f}".format(test_acc))  # 03.31.2026 CHANGED: display the RF held-out accuracy for this group
                print("Saved RF predictions to:", rf_pred_path)  # 03.31.2026 ADDED: show where the RF prediction bundle was written
                print("Saved RF model to:", rf_model_path)  # 03.31.2026 ADDED: show where the RF model file was written

            if "MLR" in exp_models:  # 03.31.2026 CHANGED: generate and save MLR predictions immediately when MLR ran
                y_pred_test_mlr = np.asarray(mlr_final.predict(X_test_full), dtype=np.int64)  # 03.31.2026 CHANGED: MLR held-out predictions for this group
                test_acc_mlr = float(np.mean(y_pred_test_mlr == y_true_test))  # 03.31.2026 CHANGED: MLR held-out accuracy for this group
                mlr_models[gname] = mlr_final  # 03.31.2026 CHANGED: keep the fitted MLR model in memory for later cells
                y_test_pred_mlr_by_group[gname] = y_pred_test_mlr  # 03.31.2026 CHANGED: keep MLR held-out predictions in memory for later cells
                test_scores_mlr_by_group[gname] = test_acc_mlr  # 03.31.2026 CHANGED: keep MLR held-out accuracy in memory for later cells
                mlr_pred_path = exp_prediction_dir / "test_predictions_MLR_{}.npz".format(gname)  # 03.31.2026 ADDED: experiment-local MLR prediction bundle path
                np.savez(str(mlr_pred_path), test_idx=np.asarray(exp_test_idx, dtype=np.int64), y_true=np.asarray(y_true_test, dtype=np.int64), y_pred=np.asarray(y_pred_test_mlr, dtype=np.int64), class_order=np.asarray(exp_class_order, dtype=object))  # 03.31.2026 ADDED: save MLR predictions immediately after they are generated
                mlr_model_path = exp_modeldir / "MLR_model_{}.joblib".format(gname)  # 03.31.2026 ADDED: experiment-local MLR model path
                joblib.dump(mlr_final, mlr_model_path)  # 03.31.2026 ADDED: save the fitted MLR model immediately after it is generated
                exp_saved_model_rows.append({"model": "MLR", "group": gname, "path": str(mlr_model_path)})  # 03.31.2026 ADDED: record the saved MLR model in the experiment manifest
                print("FINAL test acc (MLR): {:.4f}".format(test_acc_mlr))  # 03.31.2026 CHANGED: display the MLR held-out accuracy for this group
                print("Saved MLR predictions to:", mlr_pred_path)  # 03.31.2026 ADDED: show where the MLR prediction bundle was written
                print("Saved MLR model to:", mlr_model_path)  # 03.31.2026 ADDED: show where the MLR model file was written

            if "SVM" in exp_models:  # 03.31.2026 CHANGED: generate and save SVM predictions immediately when SVM ran
                y_pred_test_svm = np.asarray(svm_final.predict(X_test_full), dtype=np.int64)  # 03.31.2026 CHANGED: SVM held-out predictions for this group
                test_acc_svm = float(np.mean(y_pred_test_svm == y_true_test))  # 03.31.2026 CHANGED: SVM held-out accuracy for this group
                svm_models[gname] = svm_final  # 03.31.2026 CHANGED: keep the fitted SVM model in memory for later cells
                y_test_pred_svm_by_group[gname] = y_pred_test_svm  # 03.31.2026 CHANGED: keep SVM held-out predictions in memory for later cells
                test_scores_svm_by_group[gname] = test_acc_svm  # 03.31.2026 CHANGED: keep SVM held-out accuracy in memory for later cells
                svm_pred_path = exp_prediction_dir / "test_predictions_SVM_{}.npz".format(gname)  # 03.31.2026 ADDED: experiment-local SVM prediction bundle path
                np.savez(str(svm_pred_path), test_idx=np.asarray(exp_test_idx, dtype=np.int64), y_true=np.asarray(y_true_test, dtype=np.int64), y_pred=np.asarray(y_pred_test_svm, dtype=np.int64), class_order=np.asarray(exp_class_order, dtype=object))  # 03.31.2026 ADDED: save SVM predictions immediately after they are generated
                svm_model_path = exp_modeldir / "SVM_model_{}.joblib".format(gname)  # 03.31.2026 ADDED: experiment-local SVM model path
                joblib.dump(svm_final, svm_model_path)  # 03.31.2026 ADDED: save the fitted SVM model immediately after it is generated
                exp_saved_model_rows.append({"model": "SVM", "group": gname, "path": str(svm_model_path)})  # 03.31.2026 ADDED: record the saved SVM model in the experiment manifest
                print("FINAL test acc (SVM): {:.4f}".format(test_acc_svm))  # 03.31.2026 CHANGED: display the SVM held-out accuracy for this group
                print("Saved SVM predictions to:", svm_pred_path)  # 03.31.2026 ADDED: show where the SVM prediction bundle was written
                print("Saved SVM model to:", svm_model_path)  # 03.31.2026 ADDED: show where the SVM model file was written

            del X_test_full  # 03.31.2026 CHANGED: release the full-test memmap handle before deleting its file
            gc.collect()  # 03.31.2026 CHANGED: encourage cleanup before the next group build
            if Path(xtest_path).exists():  # 03.31.2026 CHANGED: guard full-test memmap deletion in case the file is already gone
                Path(xtest_path).unlink()  # 03.31.2026 CHANGED: delete the full-test memmap file now

        print("\nSummary (held-out test acc) for {}:".format(exp_name))  # 03.31.2026 ADDED: concise experiment-level score summary after all groups finish
        for gname in exp_groups:  # 03.31.2026 CHANGED: summarize only the groups that were requested in this experiment
            parts = ["{:>15s}:".format(gname)]  # 03.31.2026 CHANGED: start one formatted summary line for this group
            if "RF" in exp_models:  # 03.31.2026 CHANGED: add RF summary only when RF ran in this experiment
                parts.append("RF={:.4f} (CV {:.4f})".format(float(test_scores_by_group[gname]), float(np.mean(cv_scores_by_group[gname]))))  # 03.31.2026 CHANGED: append the RF held-out and CV mean for this group
            if "MLR" in exp_models:  # 03.31.2026 CHANGED: add MLR summary only when MLR ran in this experiment
                parts.append("MLR={:.4f} (CV {:.4f})".format(float(test_scores_mlr_by_group[gname]), float(np.mean(cv_scores_mlr_by_group[gname]))))  # 03.31.2026 CHANGED: append the MLR held-out and CV mean for this group
            if "SVM" in exp_models:  # 03.31.2026 CHANGED: add SVM summary only when SVM ran in this experiment
                parts.append("SVM={:.4f} (CV {:.4f})".format(float(test_scores_svm_by_group[gname]), float(np.mean(cv_scores_svm_by_group[gname]))))  # 03.31.2026 CHANGED: append the SVM held-out and CV mean for this group
            print(" ".join(parts))  # 03.31.2026 CHANGED: print the final summary line for this group

        saved_models_manifest_df = pd.DataFrame(exp_saved_model_rows)  # 03.31.2026 ADDED: experiment-specific saved-model manifest dataframe
        saved_models_manifest_df.to_csv(exp_outdir / "saved_models_manifest.csv", index=False)  # 03.31.2026 ADDED: save the experiment-specific model manifest immediately
        exp_context = {  # 03.31.2026 ADDED: experiment context saved for later browsing and interpretation
            "RUN_DIR": str(RUN_DIR),  # 03.31.2026 ADDED: dataset run folder used by this experiment
            "ZARR_PATH": str(ZARR_PATH),  # 03.31.2026 ADDED: dataset.zarr path used by this experiment
            "META_PATH": str(META_PATH),  # 03.31.2026 ADDED: metadata.csv path used by this experiment
            "OUTDIR": str(exp_outdir),  # 03.31.2026 ADDED: experiment output folder path
            "EXPERIMENT_NAME": str(exp_name),  # 03.31.2026 ADDED: saved experiment name
            "EXPERIMENT_INDEX": int(exp_idx),  # 03.31.2026 ADDED: 1-based experiment index inside the batch run
            "SEED": int(exp_seed),  # 03.31.2026 ADDED: experiment seed used for the split and model fits
            "MERGE_GRAY_CLASSES": bool(exp_merge_gray_classes),  # 03.31.2026 ADDED: whether gray classes were merged in this experiment
            "SUBSET_MAX_PER_CLASS": None if exp_subset_max_per_class is None else int(exp_subset_max_per_class),  # 03.31.2026 ADDED: optional per-class subset size used by this experiment
            "CLASS_ORDER": list(exp_class_order),  # 03.31.2026 ADDED: int-to-label mapping for this experiment
            "RESPONSE_KEYS": list(keys_order),  # 03.31.2026 ADDED: canonical response-key order in outs_fill
            "CHANNEL_GROUPS": {k: list(v) for k, v in CHANNEL_GROUPS.items()},  # 03.31.2026 ADDED: all available channel-group definitions
            "EXPERIMENT_GROUPS": list(exp_groups),  # 03.31.2026 ADDED: groups actually used by this experiment
            "EXPERIMENT_MODELS": list(exp_models),  # 03.31.2026 ADDED: models actually used by this experiment
            "DOWNSAMPLE_MAPS": bool(DOWNSAMPLE_MAPS),  # 03.31.2026 ADDED: whether response maps were downsampled before flattening
            "TARGET_HW": tuple(int(x) for x in TARGET_HW),  # 03.31.2026 ADDED: feature-map size used after downsampling
            "DOWNSAMPLE_ORDER": int(DOWNSAMPLE_ORDER),  # 03.31.2026 ADDED: interpolation order used for downsampling
            "H0W0K": {"H0": int(H0), "W0": int(W0), "H": int(H), "W": int(W), "K": int(K)},  # 03.31.2026 ADDED: source and feature map shapes for this experiment
            "RF_PARAMS": {"N_EST": int(N_EST), "MAX_DEPTH": int(MAX_DEPTH), "RF_N_JOBS": int(RF_N_JOBS)},  # 03.31.2026 ADDED: RF hyperparameters used by this experiment
            "MLR_PARAMS": {"MLR_MAX_ITER": int(MLR_MAX_ITER), "MLR_C": float(MLR_C), "MLR_TOL": float(MLR_TOL)},  # 03.31.2026 ADDED: MLR hyperparameters used by this experiment
            "SVM_PARAMS": {"SVM_KERNEL": str(SVM_KERNEL), "SVM_C": float(SVM_C), "SVM_GAMMA": str(SVM_GAMMA), "SVM_PROBABILITY": bool(SVM_PROBABILITY)},  # 03.31.2026 ADDED: SVM hyperparameters used by this experiment
            "SPLIT_PATH": str(exp_split_path),  # 03.31.2026 ADDED: experiment-local split copy path
            "SPLIT_CACHE_PATH": str(exp_split_cache_path),  # 03.31.2026 ADDED: reusable split-cache path for this experiment configuration
        }  # 03.31.2026 ADDED: end experiment context dictionary
        exp_context_path.write_text(json.dumps(exp_context, indent=2, sort_keys=True))  # 03.31.2026 ADDED: save the experiment context immediately after training

        exp_result = {  # 03.31.2026 ADDED: in-memory record for this experiment used by later cells
            "name": exp_name,  # 03.31.2026 ADDED: experiment name
            "outdir": exp_outdir,  # 03.31.2026 ADDED: experiment output folder
            "modeldir": exp_modeldir,  # 03.31.2026 ADDED: experiment model folder
            "prediction_dir": exp_prediction_dir,  # 03.31.2026 ADDED: experiment prediction bundle folder
            "split_path": exp_split_path,  # 03.31.2026 ADDED: experiment-local split copy path
            "split_cache_path": exp_split_cache_path,  # 03.31.2026 ADDED: reusable split-cache path
            "context_path": exp_context_path,  # 03.31.2026 ADDED: experiment context JSON path
            "seed": exp_seed,  # 03.31.2026 ADDED: experiment seed
            "merge_gray_classes": exp_merge_gray_classes,  # 03.31.2026 ADDED: experiment gray-merge flag
            "subset_max_per_class": exp_subset_max_per_class,  # 03.31.2026 ADDED: experiment per-class subset size
            "groups": list(exp_groups),  # 03.31.2026 ADDED: groups used by this experiment
            "models_to_run": list(exp_models),  # 03.31.2026 ADDED: models used by this experiment
            "meta": exp_meta,  # 03.31.2026 ADDED: experiment metadata after optional label merging and subsetting
            "y_series": exp_y_series,  # 03.31.2026 ADDED: experiment label integers indexed by stim_ID
            "class_order": list(exp_class_order),  # 03.31.2026 ADDED: experiment class order
            "train_idx": np.asarray(exp_train_idx, dtype=np.int64),  # 03.31.2026 ADDED: train stimulus ids for this experiment
            "test_idx": np.asarray(exp_test_idx, dtype=np.int64),  # 03.31.2026 ADDED: test stimulus ids for this experiment
            "rf_models": rf_models,  # 03.31.2026 ADDED: fitted RF models keyed by group
            "mlr_models": mlr_models,  # 03.31.2026 ADDED: fitted MLR models keyed by group
            "svm_models": svm_models,  # 03.31.2026 ADDED: fitted SVM models keyed by group
            "cv_scores_by_group": cv_scores_by_group,  # 03.31.2026 ADDED: RF CV scores keyed by group
            "cv_scores_mlr_by_group": cv_scores_mlr_by_group,  # 03.31.2026 ADDED: MLR CV scores keyed by group
            "cv_scores_svm_by_group": cv_scores_svm_by_group,  # 03.31.2026 ADDED: SVM CV scores keyed by group
            "test_scores_by_group": test_scores_by_group,  # 03.31.2026 ADDED: RF held-out scores keyed by group
            "test_scores_mlr_by_group": test_scores_mlr_by_group,  # 03.31.2026 ADDED: MLR held-out scores keyed by group
            "test_scores_svm_by_group": test_scores_svm_by_group,  # 03.31.2026 ADDED: SVM held-out scores keyed by group
            "y_test_pred_by_group": y_test_pred_by_group,  # 03.31.2026 ADDED: RF held-out predictions keyed by group
            "y_test_pred_mlr_by_group": y_test_pred_mlr_by_group,  # 03.31.2026 ADDED: MLR held-out predictions keyed by group
            "y_test_pred_svm_by_group": y_test_pred_svm_by_group,  # 03.31.2026 ADDED: SVM held-out predictions keyed by group
            "saved_models_manifest_df": saved_models_manifest_df,  # 03.31.2026 ADDED: experiment-specific saved-model manifest dataframe
        }  # 03.31.2026 ADDED: end in-memory experiment result record

        EXPERIMENT_RESULTS.append(exp_result)  # 03.31.2026 ADDED: append this experiment result to the batch result list
        EXPERIMENT_RESULTS_BY_NAME[exp_name] = exp_result  # 03.31.2026 ADDED: save a direct lookup from experiment name to result record

    print("\nSaved experiment folders in:", OUTDIR)  # 03.31.2026 ADDED: show the batch root after all experiments finish training
    print("Finished experiments:", [r["name"] for r in EXPERIMENT_RESULTS])  # 03.31.2026 ADDED: list the experiment names completed in this notebook run


=== Classic4 (channels=4) ===


Classic4 fold0 X_train:   0%|          | 0/1387 [00:00<?, ?it/s]

Classic4 fold0 X_val:   0%|          | 0/1388 [00:00<?, ?it/s]

In [ ]:
# Cell 7 — Evaluate (confusion matrices + classification reports)  # 03.31.2026 CHANGED: save per-experiment outputs and combined summaries across all experiments

if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: display saved summaries instead of recomputing evaluation artifacts
    if COMBINED_EVAL_SUMMARY_DF is not None:  # 03.31.2026 ADDED: show the saved combined summary when it exists
        display(COMBINED_EVAL_SUMMARY_DF)  # 03.31.2026 ADDED: display the saved combined summary dataframe inline
    else:  # 03.31.2026 ADDED: fall back to any per-experiment summary files that were discovered on disk
        loaded_rows = []  # 03.31.2026 ADDED: collect per-experiment summary rows from saved experiment folders
        for exp_name in SAVED_EXPERIMENT_NAMES:  # 03.31.2026 ADDED: scan each saved experiment folder for an eval summary
            exp_dir = SAVED_EXPERIMENT_DIRS[exp_name]  # 03.31.2026 ADDED: folder path for this saved experiment
            exp_summary_path = exp_dir / "eval_summary.csv"  # 03.31.2026 ADDED: expected per-experiment eval summary path
            if exp_summary_path.exists():  # 03.31.2026 ADDED: load the per-experiment summary when present
                exp_df = pd.read_csv(exp_summary_path)  # 03.31.2026 ADDED: read the per-experiment summary into a dataframe
                loaded_rows.append(exp_df)  # 03.31.2026 ADDED: collect the per-experiment summary dataframe

        if len(loaded_rows) > 0:  # 03.31.2026 ADDED: show the concatenated saved summaries when at least one exists
            COMBINED_EVAL_SUMMARY_DF = pd.concat(loaded_rows, ignore_index=True)  # 03.31.2026 ADDED: combine all saved per-experiment summaries into one dataframe
            display(COMBINED_EVAL_SUMMARY_DF)  # 03.31.2026 ADDED: display the combined saved per-experiment summaries inline
        else:  # 03.31.2026 ADDED: stop clearly when no saved summaries were found
            print("No saved evaluation summary CSV files were found in:", OUTDIR)  # 03.31.2026 ADDED: clear message for an empty saved-results folder
else:  # 03.31.2026 CHANGED: create fresh evaluation artifacts from the in-memory experiment results
    SHOW_CM = True  # 03.31.2026 CHANGED: keep confusion-matrix saving enabled for fresh experiment runs
    SHOW_CM_INLINE = True  # 03.31.2026 CHANGED: display the annotated confusion matrix inline during a fresh run
    CM_CMAP = plt.get_cmap("berlin")  # 03.31.2026 CHANGED: use the Berlin colormap for all confusion matrices
    CM_VMIN = 0.0  # 03.31.2026 CHANGED: fixed lower color bound across all experiments
    CM_VMAX = 100.0  # 03.31.2026 CHANGED: fixed upper color bound across all experiments
    CM_DECIMALS = 1  # 03.31.2026 CHANGED: show row-normalized percentages with one decimal place
    combined_rows = []  # 03.31.2026 ADDED: collect one summary row per experiment/model/group evaluation
    combined_accuracy_rows = []  # 03.31.2026 ADDED: collect the compact combined accuracy table requested by the user

    def compute_cm_percentages(cm_counts):  # 03.31.2026 ADDED: convert raw confusion-matrix counts into row-normalized percentages
        row_totals = cm_counts.sum(axis=1).astype(np.int64)  # 03.31.2026 ADDED: total sample count per true class
        cm_pct = np.zeros_like(cm_counts, dtype=np.float64)  # 03.31.2026 ADDED: allocate the row-normalized percentage matrix
        for i in range(cm_counts.shape[0]):  # 03.31.2026 ADDED: normalize one true-class row at a time
            if int(row_totals[i]) > 0:  # 03.31.2026 ADDED: skip empty rows safely
                cm_pct[i, :] = (cm_counts[i, :].astype(np.float64) / float(row_totals[i])) * 100.0  # 03.31.2026 ADDED: row-normalized percent values for this true class
        return cm_pct, row_totals  # 03.31.2026 ADDED: return both percentages and row totals for later saving

    def save_cm_png(cm_pct, class_names, outdir, model_name, gname, acc, variant, annotate_numbers, show_inline=False):  # 03.31.2026 ADDED: save one row-normalized confusion-matrix PNG for one experiment/model/group
        cm_path = outdir / "{}_cm_pct_{}_{}_acc{:.3f}.png".format(model_name, gname, variant, acc)  # 03.31.2026 ADDED: deterministic confusion-matrix PNG filename
        fig, ax = plt.subplots(figsize=(7, 7))  # 03.31.2026 ADDED: slightly larger figure so labels fit clearly
        im = ax.imshow(cm_pct, cmap=CM_CMAP, vmin=CM_VMIN, vmax=CM_VMAX)  # 03.31.2026 ADDED: fixed Berlin color scale from 0 to 100
        ax.set_xticks(np.arange(len(class_names)))  # 03.31.2026 ADDED: one x tick per predicted class
        ax.set_yticks(np.arange(len(class_names)))  # 03.31.2026 ADDED: one y tick per true class
        ax.set_xticklabels(class_names, rotation=45, ha="right", rotation_mode="anchor")  # 03.31.2026 ADDED: readable predicted-class labels
        ax.set_yticklabels(class_names)  # 03.31.2026 ADDED: readable true-class labels
        ax.tick_params(axis="x", labelsize=11)  # 03.31.2026 ADDED: keep x-axis labels legible
        ax.tick_params(axis="y", labelsize=11)  # 03.31.2026 ADDED: keep y-axis labels legible
        ax.set_xlabel("Predicted label")  # 03.31.2026 ADDED: label the x axis clearly
        ax.set_ylabel("True label")  # 03.31.2026 ADDED: label the y axis clearly
        ax.set_title("{} / {} final test, row-normalized".format(model_name, gname))  # 03.31.2026 ADDED: descriptive confusion-matrix title
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)  # 03.31.2026 ADDED: add a colorbar for the percentage scale
        cbar.set_label("Percent of true class")  # 03.31.2026 ADDED: label the colorbar clearly

        if annotate_numbers:  # 03.31.2026 ADDED: optionally write percentage values inside the confusion-matrix cells
            for i in range(cm_pct.shape[0]):  # 03.31.2026 ADDED: loop over true-class rows
                for j in range(cm_pct.shape[1]):  # 03.31.2026 ADDED: loop over predicted-class columns
                    value = float(cm_pct[i, j])  # 03.31.2026 ADDED: current cell percentage value
                    text_color = "white" if value >= 50.0 else "black"  # 03.31.2026 ADDED: keep annotation text readable on dark cells
                    ax.text(j, i, "{:.1f}".format(value), ha="center", va="center", color=text_color, fontsize=10)  # 03.31.2026 ADDED: write one percentage annotation inside this cell

        fig.subplots_adjust(left=0.22, bottom=0.20, right=0.92, top=0.90)  # 03.31.2026 ADDED: reserve space so labels are not clipped
        fig.savefig(cm_path, dpi=150)  # 03.31.2026 ADDED: save the confusion-matrix PNG to disk
        if show_inline:  # 03.31.2026 ADDED: optionally display the confusion matrix in the notebook output
            plt.show()  # 03.31.2026 ADDED: show the confusion matrix inline
        plt.close(fig)  # 03.31.2026 ADDED: close the figure to keep memory predictable
        return cm_path  # 03.31.2026 ADDED: return the saved confusion-matrix PNG path

    def save_cm_table(cm_counts, cm_pct, row_totals, class_names, outdir, model_name, gname, acc):  # 03.31.2026 ADDED: save one confusion-matrix number table as CSV
        cm_rows = []  # 03.31.2026 ADDED: collect one row per confusion-matrix cell

        for i, true_label in enumerate(class_names):  # 03.31.2026 ADDED: loop over true classes
            for j, pred_label in enumerate(class_names):  # 03.31.2026 ADDED: loop over predicted classes
                cm_rows.append({"model": model_name, "group": gname, "true_label": true_label, "pred_label": pred_label, "is_error": bool(i != j), "raw_count": int(cm_counts[i, j]), "true_class_total": int(row_totals[i]), "percent_of_true_class": float(cm_pct[i, j])})  # 03.31.2026 ADDED: save the numeric contents of this confusion-matrix cell

        cm_table_path = outdir / "{}_cm_table_{}_acc{:.3f}.csv".format(model_name, gname, acc)  # 03.31.2026 ADDED: deterministic confusion-matrix CSV filename
        pd.DataFrame(cm_rows).to_csv(cm_table_path, index=False)  # 03.31.2026 ADDED: write the confusion-matrix numbers to CSV
        return cm_table_path  # 03.31.2026 ADDED: return the saved confusion-matrix CSV path

    def save_report_and_cm(y_true, y_pred, class_names, outdir, model_name, gname):  # 03.31.2026 ADDED: save one text report plus confusion-matrix artifacts for one experiment/model/group
        labels_int = list(range(len(class_names)))  # 03.31.2026 ADDED: integer labels aligned to the provided class_names list
        acc = float(np.mean(y_pred == y_true))  # 03.31.2026 ADDED: held-out test accuracy for this experiment/model/group
        cm_counts = confusion_matrix(y_true, y_pred, labels=labels_int)  # 03.31.2026 ADDED: raw confusion-matrix counts
        cm_pct, row_totals = compute_cm_percentages(cm_counts)  # 03.31.2026 ADDED: row-normalized confusion-matrix percentages and row totals
        report = classification_report(y_true, y_pred, labels=labels_int, target_names=class_names, digits=4, zero_division=0)  # 03.31.2026 ADDED: classification report text for this experiment/model/group
        rep_path = outdir / "{}_report_{}.txt".format(model_name, gname)  # 03.31.2026 ADDED: deterministic classification-report filename
        rep_path.write_text(report)  # 03.31.2026 ADDED: save the classification report immediately
        cm_table_path = save_cm_table(cm_counts, cm_pct, row_totals, class_names, outdir, model_name, gname, acc)  # 03.31.2026 ADDED: save the confusion-matrix number table immediately
        cm_annotated_path = None  # 03.31.2026 ADDED: default annotated confusion-matrix path
        cm_blank_path = None  # 03.31.2026 ADDED: default blank confusion-matrix path

        if SHOW_CM:  # 03.31.2026 ADDED: save confusion-matrix PNGs only when confusion-matrix saving is enabled
            cm_annotated_path = save_cm_png(cm_pct, class_names, outdir, model_name, gname, acc, "annotated", True, show_inline=SHOW_CM_INLINE)  # 03.31.2026 ADDED: save and optionally show the annotated confusion matrix
            cm_blank_path = save_cm_png(cm_pct, class_names, outdir, model_name, gname, acc, "blank", False, show_inline=False)  # 03.31.2026 ADDED: save the blank confusion matrix for later figure assembly

        return acc, rep_path, cm_annotated_path, cm_blank_path, cm_table_path  # 03.31.2026 ADDED: return the saved evaluation artifact paths

    for exp_result in EXPERIMENT_RESULTS:  # 03.31.2026 ADDED: evaluate and save artifacts for every completed experiment
        exp_name = exp_result["name"]  # 03.31.2026 ADDED: current experiment name
        exp_outdir = exp_result["outdir"]  # 03.31.2026 ADDED: current experiment output folder
        exp_groups = exp_result["groups"]  # 03.31.2026 ADDED: groups used by the current experiment
        exp_models = exp_result["models_to_run"]  # 03.31.2026 ADDED: models used by the current experiment
        exp_class_order = list(exp_result["class_order"])  # 03.31.2026 ADDED: class order for the current experiment
        exp_y_true = exp_result["y_series"].loc[exp_result["test_idx"]].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: held-out test labels for the current experiment
        exp_rows = []  # 03.31.2026 ADDED: per-experiment summary rows saved to this experiment folder
        exp_pred_by_model = {"RF": exp_result["y_test_pred_by_group"], "MLR": exp_result["y_test_pred_mlr_by_group"], "SVM": exp_result["y_test_pred_svm_by_group"]}  # 03.31.2026 ADDED: unified model -> predictions lookup for this experiment
        exp_cv_by_model = {"RF": exp_result["cv_scores_by_group"], "MLR": exp_result["cv_scores_mlr_by_group"], "SVM": exp_result["cv_scores_svm_by_group"]}  # 03.31.2026 ADDED: unified model -> CV score lookup for this experiment

        for model_name in exp_models:  # 03.31.2026 ADDED: save evaluation outputs for every model that ran in this experiment
            for gname in exp_groups:  # 03.31.2026 ADDED: save evaluation outputs for every group that ran in this experiment
                y_pred = exp_pred_by_model[model_name][gname]  # 03.31.2026 ADDED: held-out predictions for this experiment/model/group
                acc, rep_path, cm_annotated_path, cm_blank_path, cm_table_path = save_report_and_cm(exp_y_true, y_pred, exp_class_order, exp_outdir, model_name, gname)  # 03.31.2026 ADDED: save reports and confusion matrices now
                cv_scores = exp_cv_by_model[model_name].get(gname, [])  # 03.31.2026 ADDED: CV scores for this experiment/model/group
                cv_mean = float(np.mean(cv_scores)) if len(cv_scores) > 0 else np.nan  # 03.31.2026 ADDED: CV mean for this experiment/model/group

                row = {"experiment": exp_name, "seed": int(exp_result["seed"]), "merge_gray_classes": bool(exp_result["merge_gray_classes"]), "subset_max_per_class": exp_result["subset_max_per_class"], "model": model_name, "group": gname, "test_acc": acc, "cv_mean": cv_mean, "report_path": str(rep_path), "cm_annotated_path": str(cm_annotated_path) if cm_annotated_path is not None else "", "cm_blank_path": str(cm_blank_path) if cm_blank_path is not None else "", "cm_table_path": str(cm_table_path)}  # 03.31.2026 ADDED: one combined summary row for this experiment/model/group
                exp_rows.append(row)  # 03.31.2026 ADDED: add this row to the per-experiment summary
                combined_rows.append(row)  # 03.31.2026 ADDED: add this row to the cross-experiment combined summary
                combined_accuracy_rows.append({"experiment": exp_name, "seed": int(exp_result["seed"]), "merge_gray_classes": bool(exp_result["merge_gray_classes"]), "subset_max_per_class": exp_result["subset_max_per_class"], "model": model_name, "group": gname, "test_acc": acc, "cv_mean": cv_mean})  # 03.31.2026 ADDED: add this compact accuracy row to the requested combined accuracy table
                pd.DataFrame(exp_rows).to_csv(exp_outdir / "eval_summary.csv", index=False)  # 03.31.2026 CHANGED: update the per-experiment summary immediately after each result is generated
                pd.DataFrame(combined_rows).to_csv(OUTDIR / "combined_eval_summary.csv", index=False)  # 03.31.2026 CHANGED: update the combined summary immediately after each result is generated
                pd.DataFrame(combined_accuracy_rows).to_csv(OUTDIR / "combined_test_accuracy_summary.csv", index=False)  # 03.31.2026 ADDED: update the combined accuracy-only table immediately after each result is generated
                print("{} / {} / {}: test acc = {:.4f} (CV mean {:.4f})".format(exp_name, model_name, gname, acc, cv_mean))  # 03.31.2026 ADDED: concise progress line for the current saved evaluation result

        exp_result["eval_summary_df"] = pd.DataFrame(exp_rows)  # 03.31.2026 ADDED: keep the per-experiment summary dataframe in memory for later cells

    COMBINED_EVAL_SUMMARY_DF = pd.DataFrame(combined_rows)  # 03.31.2026 ADDED: in-memory combined summary across all experiments
    COMBINED_TEST_ACCURACY_DF = pd.DataFrame(combined_accuracy_rows)  # 03.31.2026 ADDED: in-memory compact combined accuracy table across all experiments
    display(COMBINED_EVAL_SUMMARY_DF)  # 03.31.2026 ADDED: display the combined summary after all experiments finish evaluating

In [ ]:
# Cell 8 — Save misclassified examples (metadata + images)  # 03.31.2026 CHANGED: save misclassified outputs inside each experiment folder

if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: skip new misclassified saving when browsing a saved results folder
    print("[USE_LOADED_RESULTS] Skipping Cell 8 (misclassified saving).")  # 03.31.2026 CHANGED: clear status message for saved-results mode
    print("Misclassified CSVs / image folders (if present) are already in:", OUTDIR)  # 03.31.2026 CHANGED: remind the user where saved misclassified outputs live
else:  # 03.31.2026 CHANGED: save fresh misclassified outputs for every completed experiment
    for exp_result in EXPERIMENT_RESULTS:  # 03.31.2026 ADDED: save misclassified outputs for every experiment that just ran
        exp_name = exp_result["name"]  # 03.31.2026 ADDED: current experiment name
        exp_outdir = exp_result["outdir"]  # 03.31.2026 ADDED: current experiment output folder
        exp_groups = exp_result["groups"]  # 03.31.2026 ADDED: groups used by the current experiment
        exp_models = exp_result["models_to_run"]  # 03.31.2026 ADDED: models used by the current experiment
        exp_class_order = list(exp_result["class_order"])  # 03.31.2026 ADDED: class order used by the current experiment
        exp_y_true = exp_result["y_series"].loc[exp_result["test_idx"]].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: held-out test labels aligned to the experiment test ids
        exp_test_idx = np.asarray(exp_result["test_idx"], dtype=np.int64)  # 03.31.2026 ADDED: held-out test stimulus ids for the current experiment
        exp_meta = exp_result["meta"]  # 03.31.2026 ADDED: experiment metadata after optional label merging and subsetting
        exp_pred_by_model = {"RF": exp_result["y_test_pred_by_group"], "MLR": exp_result["y_test_pred_mlr_by_group"], "SVM": exp_result["y_test_pred_svm_by_group"]}  # 03.31.2026 ADDED: unified model -> predictions lookup for this experiment

        print("\nSaving misclassified examples for experiment:", exp_name)  # 03.31.2026 ADDED: clear experiment header for misclassified saving

        for model_name in exp_models:  # 03.31.2026 ADDED: save misclassified outputs for every model that ran in this experiment
            for gname in exp_groups:  # 03.31.2026 ADDED: save misclassified outputs for every group that ran in this experiment
                y_pred = np.asarray(exp_pred_by_model[model_name][gname], dtype=np.int64)  # 03.31.2026 ADDED: held-out predictions for the current experiment/model/group
                wrong_mask = (y_pred != exp_y_true)  # 03.31.2026 CHANGED: misclassification mask on the current experiment test set
                wrong_ids = exp_test_idx[wrong_mask]  # 03.31.2026 CHANGED: misclassified stimulus ids for the current experiment/model/group
                print("{} / {} / {}: misclassified {}/{}".format(exp_name, model_name, gname, int(len(wrong_ids)), int(len(exp_test_idx))))  # 03.31.2026 CHANGED: concise misclassification count for the current result

                df = exp_meta.loc[wrong_ids].copy()  # 03.31.2026 CHANGED: metadata rows for the misclassified stimuli only
                df["experiment"] = exp_name  # 03.31.2026 ADDED: save the experiment name in the misclassified metadata CSV
                df["model"] = model_name  # 03.31.2026 CHANGED: save the model name in the misclassified metadata CSV
                df["group"] = gname  # 03.31.2026 CHANGED: save the group name in the misclassified metadata CSV
                df["true_label"] = [exp_class_order[int(t)] for t in exp_y_true[wrong_mask]]  # 03.31.2026 CHANGED: save experiment-specific true labels in the misclassified metadata CSV
                df["pred_label"] = [exp_class_order[int(p)] for p in y_pred[wrong_mask]]  # 03.31.2026 CHANGED: save experiment-specific predicted labels in the misclassified metadata CSV
                df.to_csv(exp_outdir / ("misclassified_{}_{}.csv".format(model_name, gname)), index=True)  # 03.31.2026 CHANGED: save the misclassified metadata CSV inside the experiment folder

                save_dir = exp_outdir / ("misclassified_imgs_{}_{}".format(model_name, gname))  # 03.31.2026 CHANGED: save misclassified images inside the experiment folder
                save_dir.mkdir(parents=True, exist_ok=True)  # 03.31.2026 CHANGED: make sure the misclassified image folder exists before saving PNGs

                for sid in wrong_ids:  # 03.31.2026 CHANGED: save one PNG per misclassified stimulus id
                    img = np.asarray(imgs_z[int(sid)], dtype=np.float32)  # 03.31.2026 CHANGED: load the source RGB image for this misclassified stimulus
                    img = np.clip(img, 0.0, 1.0)  # 03.31.2026 CHANGED: clamp the image to display-safe RGB values
                    out_path = save_dir / ("stim_{:06d}.png".format(int(sid)))  # 03.31.2026 CHANGED: deterministic PNG filename for this misclassified stimulus
                    plt.imsave(out_path, img)  # 03.31.2026 CHANGED: save the misclassified stimulus image as a PNG

    print("Done. Outputs in:", OUTDIR)  # 03.31.2026 CHANGED: confirm the batch root that now contains all experiment outputs

In [ ]:
# Cell 9 — Interactive viewer for misclassified images (works for fresh runs OR saved-results folders)  # 03.31.2026 CHANGED: add experiment browsing for the new multi-experiment save layout

from pathlib import Path  # 02.26.2026 CHANGED: safer path handling inside the viewer
import matplotlib.pyplot as plt  # 02.26.2026 CHANGED: display source or saved PNG images
from IPython.display import display, clear_output  # 02.26.2026 CHANGED: interactive notebook display helpers
import ipywidgets as widgets  # 02.26.2026 CHANGED: interactive dropdowns and buttons

if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: pull experiment/model/group options from a saved batch folder
    experiment_options = list(SAVED_EXPERIMENT_NAMES)  # 03.31.2026 ADDED: saved experiment names discovered on disk
else:  # 03.31.2026 CHANGED: pull experiment/model/group options from the fresh in-memory experiment results
    if "EXPERIMENT_RESULTS" not in globals() or len(EXPERIMENT_RESULTS) == 0:  # 03.31.2026 ADDED: fail clearly when the training/evaluation cells have not been run yet
        raise ValueError("No experiments are available to view. Run Cells 6–8 first, or load a saved results directory.")  # 03.31.2026 ADDED: clear viewer error for missing experiment results
    experiment_options = [r["name"] for r in EXPERIMENT_RESULTS]  # 03.31.2026 ADDED: fresh-run experiment names for the experiment dropdown

if len(experiment_options) == 0:  # 03.31.2026 ADDED: fail clearly when no experiment folders or experiment results were found
    raise ValueError("No experiments are available to view.")  # 03.31.2026 ADDED: clear viewer error for an empty experiment list

def _get_model_options(exp_name):  # 03.31.2026 ADDED: return the available model names for the selected experiment
    if USE_LOADED_RESULTS:  # 03.31.2026 ADDED: read saved model options from the saved-results discovery dictionaries
        return list(SAVED_MODELS_BY_EXPERIMENT.get(exp_name, []))  # 03.31.2026 ADDED: saved model options for the selected experiment
    return list(EXPERIMENT_RESULTS_BY_NAME[exp_name]["models_to_run"])  # 03.31.2026 ADDED: fresh-run model options for the selected experiment

def _get_group_options(exp_name):  # 03.31.2026 ADDED: return the available group names for the selected experiment
    if USE_LOADED_RESULTS:  # 03.31.2026 ADDED: read saved group options from the saved-results discovery dictionaries
        return list(SAVED_GROUPS_BY_EXPERIMENT.get(exp_name, []))  # 03.31.2026 ADDED: saved group options for the selected experiment
    return list(EXPERIMENT_RESULTS_BY_NAME[exp_name]["groups"])  # 03.31.2026 ADDED: fresh-run group options for the selected experiment

exp_dd = widgets.Dropdown(options=experiment_options, value=experiment_options[0], description="Experiment:")  # 03.31.2026 ADDED: experiment dropdown for the new multi-experiment layout
model_dd = widgets.Dropdown(options=_get_model_options(experiment_options[0]), description="Model:")  # 03.31.2026 CHANGED: model dropdown now depends on the selected experiment
group_dd = widgets.Dropdown(options=_get_group_options(experiment_options[0]), description="Group:")  # 03.31.2026 CHANGED: group dropdown now depends on the selected experiment
prev_btn = widgets.Button(description="Prev")  # 02.26.2026 CHANGED: move to the previous misclassified item
next_btn = widgets.Button(description="Next")  # 02.26.2026 CHANGED: move to the next misclassified item
status = widgets.HTML(value="")  # 02.26.2026 CHANGED: status text above the displayed image
out = widgets.Output()  # 02.26.2026 CHANGED: output area that shows the current image

state = {"items": [], "i": 0}  # 03.31.2026 CHANGED: viewer state now tracks items for the selected experiment/model/group

def _load_saved_items(exp_name, model_name, gname):  # 03.31.2026 ADDED: build saved-viewer items from one saved experiment folder
    items = []  # 03.31.2026 ADDED: collect saved-viewer items as tuples
    exp_dir = SAVED_EXPERIMENT_DIRS[exp_name]  # 03.31.2026 ADDED: folder path for the selected saved experiment
    img_dir = exp_dir / "misclassified_imgs_{}_{}".format(model_name, gname)  # 03.31.2026 ADDED: expected saved PNG folder for the selected result
    csv_path = exp_dir / "misclassified_{}_{}.csv".format(model_name, gname)  # 03.31.2026 ADDED: expected saved metadata CSV for the selected result

    if csv_path.exists():  # 03.31.2026 ADDED: prefer the saved metadata CSV because it also stores true and predicted labels
        df = pd.read_csv(csv_path, index_col=0)  # 03.31.2026 ADDED: load the saved misclassified metadata CSV
        for sid in df.index:  # 03.31.2026 ADDED: walk through the saved misclassified stimulus ids
            try:  # 03.31.2026 ADDED: protect against non-integer CSV index values
                sid_int = int(sid)  # 03.31.2026 ADDED: normalize the saved stimulus id into an integer
            except Exception:  # 03.31.2026 ADDED: skip malformed stimulus ids cleanly
                continue  # 03.31.2026 ADDED: move on to the next saved CSV row

            png_path = img_dir / "stim_{:06d}.png".format(sid_int)  # 03.31.2026 ADDED: expected PNG path for this saved misclassified stimulus
            if not png_path.exists():  # 03.31.2026 ADDED: skip CSV rows whose PNGs are missing
                continue  # 03.31.2026 ADDED: move on to the next saved CSV row

            true_lab = str(df.loc[sid, "true_label"]) if "true_label" in df.columns else ""  # 03.31.2026 ADDED: saved true label when available
            pred_lab = str(df.loc[sid, "pred_label"]) if "pred_label" in df.columns else ""  # 03.31.2026 ADDED: saved predicted label when available
            items.append((sid_int, png_path, true_lab, pred_lab))  # 03.31.2026 ADDED: save one full saved-viewer item

        return items  # 03.31.2026 ADDED: return the saved-viewer items built from the metadata CSV

    if img_dir.exists():  # 03.31.2026 ADDED: fall back to listing saved PNGs when the metadata CSV is missing
        for png_path in sorted(img_dir.glob("*.png")):  # 03.31.2026 ADDED: walk through all saved PNGs in the folder
            m = re.search(r"stim_(\d+)\.png$", png_path.name)  # 03.31.2026 ADDED: parse the stimulus id out of the saved PNG filename
            sid_int = int(m.group(1)) if m else -1  # 03.31.2026 ADDED: integer stimulus id from the filename or -1 when parsing fails
            items.append((sid_int, png_path, "", ""))  # 03.31.2026 ADDED: save one PNG-only saved-viewer item

    return items  # 03.31.2026 ADDED: return the saved-viewer items discovered from disk

def _refresh_choices(*_args):  # 03.31.2026 ADDED: refresh model and group dropdown options after the experiment selection changes
    exp_name = exp_dd.value  # 03.31.2026 ADDED: currently selected experiment name
    model_options = _get_model_options(exp_name)  # 03.31.2026 ADDED: model options valid for the selected experiment
    group_options = _get_group_options(exp_name)  # 03.31.2026 ADDED: group options valid for the selected experiment
    model_dd.options = model_options  # 03.31.2026 ADDED: update the model dropdown choices for this experiment
    group_dd.options = group_options  # 03.31.2026 ADDED: update the group dropdown choices for this experiment
    model_dd.value = model_options[0] if len(model_options) > 0 else None  # 03.31.2026 ADDED: choose a valid model after the options update
    group_dd.value = group_options[0] if len(group_options) > 0 else None  # 03.31.2026 ADDED: choose a valid group after the options update
    _refresh_items()  # 03.31.2026 ADDED: refresh the item list for the new experiment/model/group selection

def _refresh_items(*_args):  # 03.31.2026 CHANGED: refresh the misclassified item list for the selected experiment/model/group
    exp_name = exp_dd.value  # 03.31.2026 ADDED: currently selected experiment name
    model_name = model_dd.value  # 03.31.2026 CHANGED: currently selected model name
    gname = group_dd.value  # 03.31.2026 CHANGED: currently selected group name

    if model_name is None or gname is None:  # 03.31.2026 ADDED: handle empty dropdown states safely
        state["items"] = []  # 03.31.2026 ADDED: no items are available when model or group is missing
        state["i"] = 0  # 03.31.2026 ADDED: reset the current item pointer when no items are available
        _show_current()  # 03.31.2026 ADDED: refresh the viewer output for the empty state
        return  # 03.31.2026 ADDED: stop here for the empty dropdown state

    if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: read saved misclassified items from disk when browsing a saved results folder
        state["items"] = _load_saved_items(exp_name, model_name, gname)  # 03.31.2026 CHANGED: saved-viewer items for the selected experiment/model/group
        state["i"] = 0  # 03.31.2026 CHANGED: reset the current item pointer after every selection change
        _show_current()  # 03.31.2026 CHANGED: refresh the viewer output immediately
        return  # 03.31.2026 CHANGED: no in-memory prediction work is needed in saved-results mode

    exp_result = EXPERIMENT_RESULTS_BY_NAME[exp_name]  # 03.31.2026 ADDED: in-memory result record for the selected experiment
    y_true = exp_result["y_series"].loc[exp_result["test_idx"]].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: held-out test labels for the selected experiment
    pred_lookup = {"RF": exp_result["y_test_pred_by_group"], "MLR": exp_result["y_test_pred_mlr_by_group"], "SVM": exp_result["y_test_pred_svm_by_group"]}  # 03.31.2026 ADDED: unified model -> predictions lookup for the selected experiment
    y_pred = pred_lookup[model_name][gname]  # 03.31.2026 ADDED: held-out predictions for the selected experiment/model/group
    wrong_mask = (y_pred != y_true)  # 03.31.2026 CHANGED: misclassification mask for the selected experiment/model/group
    wrong_ids = np.asarray(exp_result["test_idx"], dtype=np.int64)[wrong_mask]  # 03.31.2026 CHANGED: misclassified stimulus ids for the selected experiment/model/group
    state["items"] = [int(s) for s in wrong_ids]  # 03.31.2026 CHANGED: store the current experiment's misclassified stimulus ids
    state["i"] = 0  # 03.31.2026 CHANGED: reset the current item pointer after every selection change
    _show_current()  # 03.31.2026 CHANGED: refresh the viewer output immediately

def _show_current():  # 03.31.2026 CHANGED: display the current misclassified item for the selected experiment/model/group
    items = state["items"]  # 03.31.2026 CHANGED: current list of saved or in-memory misclassified items
    i = int(state["i"])  # 03.31.2026 CHANGED: current item position inside the item list
    exp_name = exp_dd.value  # 03.31.2026 ADDED: currently selected experiment name
    model_name = model_dd.value  # 03.31.2026 CHANGED: currently selected model name
    gname = group_dd.value  # 03.31.2026 CHANGED: currently selected group name

    with out:  # 03.31.2026 CHANGED: redraw the viewer output area
        clear_output(wait=True)  # 03.31.2026 CHANGED: refresh the output area cleanly

        if len(items) == 0:  # 03.31.2026 CHANGED: handle the empty item list clearly
            status.value = "<b>No misclassified examples for this selection.</b>"  # 03.31.2026 CHANGED: viewer status for an empty item list
            return  # 03.31.2026 CHANGED: nothing else to display for an empty item list

        i = max(0, min(i, len(items) - 1))  # 03.31.2026 CHANGED: clamp the current pointer inside the valid item range
        state["i"] = i  # 03.31.2026 CHANGED: store the clamped current pointer back into the viewer state

        if USE_LOADED_RESULTS:  # 03.31.2026 CHANGED: show a saved PNG directly when browsing a saved results folder
            sid, png_path, true_lab, pred_lab = items[i]  # 03.31.2026 CHANGED: saved item tuple for the current position
            if true_lab or pred_lab:  # 03.31.2026 CHANGED: include true and predicted labels when the saved metadata CSV provided them
                status.value = f"<b>{exp_name} / {model_name} / {gname}</b> — {i+1}/{len(items)} (stim_ID={sid}) | true=<b>{true_lab}</b> pred=<b>{pred_lab}</b>"  # 03.31.2026 CHANGED: saved-viewer status line with labels
            else:  # 03.31.2026 CHANGED: show a simpler status line when labels are unavailable
                status.value = f"<b>{exp_name} / {model_name} / {gname}</b> — {i+1}/{len(items)} (stim_ID={sid})"  # 03.31.2026 CHANGED: saved-viewer status line without labels

            img = plt.imread(str(png_path))  # 03.31.2026 CHANGED: load the saved PNG from disk
            plt.figure(figsize=(6, 6))  # 03.31.2026 CHANGED: display the saved PNG at a readable size
            plt.imshow(img)  # 03.31.2026 CHANGED: show the saved misclassified PNG
            plt.axis("off")  # 03.31.2026 CHANGED: hide axes for the saved PNG display
            plt.title(png_path.name)  # 03.31.2026 CHANGED: show the saved PNG filename above the image
            plt.show()  # 03.31.2026 CHANGED: display the saved PNG inline
            return  # 03.31.2026 CHANGED: the saved-viewer branch is done

        exp_result = EXPERIMENT_RESULTS_BY_NAME[exp_name]  # 03.31.2026 ADDED: in-memory result record for the selected experiment
        exp_class_order = list(exp_result["class_order"])  # 03.31.2026 ADDED: class order used by the selected experiment
        y_true = exp_result["y_series"].loc[exp_result["test_idx"]].to_numpy(dtype=np.int64)  # 03.31.2026 ADDED: held-out test labels for the selected experiment
        pred_lookup = {"RF": exp_result["y_test_pred_by_group"], "MLR": exp_result["y_test_pred_mlr_by_group"], "SVM": exp_result["y_test_pred_svm_by_group"]}  # 03.31.2026 ADDED: unified model -> predictions lookup for the selected experiment
        y_pred = pred_lookup[model_name][gname]  # 03.31.2026 ADDED: held-out predictions for the selected experiment/model/group
        sid = int(items[i])  # 03.31.2026 CHANGED: current misclassified stimulus id
        pos = int(np.where(np.asarray(exp_result["test_idx"], dtype=np.int64) == sid)[0][0])  # 03.31.2026 CHANGED: position of the current stimulus id inside the selected experiment test set
        true_lab = exp_class_order[int(y_true[pos])]  # 03.31.2026 CHANGED: experiment-specific true label for the current item
        pred_lab = exp_class_order[int(y_pred[pos])]  # 03.31.2026 CHANGED: experiment-specific predicted label for the current item
        status.value = f"<b>{exp_name} / {model_name} / {gname}</b> — {i+1}/{len(items)} (stim_ID={sid}) | true=<b>{true_lab}</b> pred=<b>{pred_lab}</b>"  # 03.31.2026 CHANGED: fresh-run status line with experiment/model/group and labels
        img = np.asarray(imgs_z[sid], dtype=np.float32)  # 03.31.2026 CHANGED: load the source RGB image for the current misclassified stimulus
        img = np.clip(img, 0.0, 1.0)  # 03.31.2026 CHANGED: clamp the source RGB image to display-safe values
        plt.figure(figsize=(6, 6))  # 03.31.2026 CHANGED: display the source RGB image at a readable size
        plt.imshow(img)  # 03.31.2026 CHANGED: show the source RGB image inline
        plt.axis("off")  # 03.31.2026 CHANGED: hide axes for the source RGB display
        plt.show()  # 03.31.2026 CHANGED: display the source RGB image inline

def _on_next(_):  # 03.31.2026 CHANGED: move to the next item in the current experiment/model/group selection
    state["i"] += 1  # 03.31.2026 CHANGED: increment the current item pointer
    _show_current()  # 03.31.2026 CHANGED: refresh the viewer output after moving forward

def _on_prev(_):  # 03.31.2026 CHANGED: move to the previous item in the current experiment/model/group selection
    state["i"] -= 1  # 03.31.2026 CHANGED: decrement the current item pointer
    _show_current()  # 03.31.2026 CHANGED: refresh the viewer output after moving backward

next_btn.on_click(_on_next)  # 03.31.2026 CHANGED: attach the forward button callback
prev_btn.on_click(_on_prev)  # 03.31.2026 CHANGED: attach the backward button callback
exp_dd.observe(_refresh_choices, names="value")  # 03.31.2026 ADDED: refresh model and group choices when the experiment changes
model_dd.observe(_refresh_items, names="value")  # 03.31.2026 CHANGED: refresh the item list when the model changes
group_dd.observe(_refresh_items, names="value")  # 03.31.2026 CHANGED: refresh the item list when the group changes

ui = widgets.VBox([widgets.HBox([exp_dd, model_dd, group_dd, prev_btn, next_btn]), status, out])  # 03.31.2026 CHANGED: include the new experiment dropdown in the viewer layout
display(ui)  # 03.31.2026 CHANGED: display the interactive viewer
_refresh_choices()  # 03.31.2026 ADDED: initialize model/group choices and show the first item immediately